In [33]:
import os
import shutil

# 🔹 Kaynak dosya (nereden alınacak)
src = os.path.expanduser(r"~/Desktop/projects/phishing_dataset/notebooks/top-1m.csv")

# 🔹 Hedef klasör
dst_dir = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata"
os.makedirs(dst_dir, exist_ok=True)  # klasör yoksa oluştur

# 🔹 Hedef dosya yolu
dst = os.path.join(dst_dir, "top-1m.csv")

# 🔹 Kopyalama işlemi
try:
    shutil.copy2(src, dst)
    print(f"✅ Dosya başarıyla kopyalandı:\n  {dst}")
except FileNotFoundError:
    print("⚠️ Kaynak dosya bulunamadı! Lütfen dosya yolunu kontrol et:")
    print(src)
except Exception as e:
    print("⚠️ Kopyalama sırasında hata oluştu:", e)


✅ Dosya başarıyla kopyalandı:
  C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top-1m.csv


In [34]:
# 🔹 top-1m.csv dosyasının doğru yolu
LOCAL_TOP1M = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top-1m.csv"

# 🔹 Kontrol edelim
import os
print("LOCAL_TOP1M:", LOCAL_TOP1M)
print("Dosya var mı?", os.path.exists(LOCAL_TOP1M))


LOCAL_TOP1M: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top-1m.csv
Dosya var mı? True


In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

def fetch_similarweb_top_sites():
    """
    SimilarWeb Türkiye listesi üzerinden en popüler domainleri çeker.
    """
    print("🌍 SimilarWeb'den veri çekiliyor...")
    url = "https://www.similarweb.com/tr/top-websites/turkey/"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
        print(f"⚠️ SimilarWeb isteği başarısız: HTTP {r.status_code}")
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    domains = []

    # Yeni SimilarWeb yapısına göre alan adlarını bul
    for a in soup.select("a.top-site-list__title, a.tw-ellipsis"):
        text = a.get_text(strip=True)
        if text and "." in text:
            domains.append(text.lower())

    print(f"✅ SimilarWeb: {len(domains)} site bulundu.")
    return domains


def fetch_tranco_top_sites(limit=1000):
    """
    Tranco listesinden genel popüler domainleri çeker (ülke filtresi yok).
    """
    print("🌐 Tranco listesinden veri indiriliyor...")
    url = "https://tranco-list.eu/download_latest_list"
    try:
        df = pd.read_csv(url, header=None, names=["rank", "domain"])
        print(f"✅ Tranco: {len(df)} domain indirildi.")
        return df["domain"].head(limit).tolist()
    except Exception as e:
        print("⚠️ Tranco verisi alınamadı:", e)
        return []


def merge_and_save(similarweb_domains, tranco_domains, output="top_turkey_domains.csv"):
    """
    İki listeyi birleştirip benzersiz hale getirir, CSV'ye kaydeder.
    """
    combined = pd.Series(similarweb_domains + tranco_domains).drop_duplicates().reset_index(drop=True)
    df = pd.DataFrame({"rank": range(1, len(combined) + 1), "domain": combined})
    df.to_csv(output, index=False, encoding="utf-8-sig")

    print(f"\n💾 {output} dosyası oluşturuldu ({len(df)} domain).")
    print(f"📅 Oluşturulma zamanı: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


if __name__ == "__main__":
    sw = fetch_similarweb_top_sites()
    tr = fetch_tranco_top_sites(limit=1000)
    merge_and_save(sw, tr)


🌍 SimilarWeb'den veri çekiliyor...
✅ SimilarWeb: 0 site bulundu.
🌐 Tranco listesinden veri indiriliyor...
⚠️ Tranco verisi alınamadı: HTTP Error 404: Not Found

💾 top_turkey_domains.csv dosyası oluşturuldu (0 domain).
📅 Oluşturulma zamanı: 2025-10-26 11:39:34


In [4]:
import pandas as pd
import os

# 🔹 Hedef klasör yolu
save_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata"

# 🔹 Klasör yoksa oluştur
os.makedirs(save_path, exist_ok=True)

print("📦 Tranco listesi indiriliyor...")

url = "https://tranco-list.eu/top-1m.csv.zip"

# 🔹 ZIP içindeki CSV'yi doğrudan oku
df = pd.read_csv(url, compression="zip", header=None, names=["rank", "domain"])

# 🔹 Türkiye uzantılı domainleri filtrele (.tr, .com.tr, .gov.tr, .edu.tr)
df_tr = df[df["domain"].str.endswith((".tr", ".com.tr", ".gov.tr", ".edu.tr"))]

# 🔹 Kaydetme yolu
output_file = os.path.join(save_path, "top_1000_turkey_tranco.csv")

# 🔹 İlk 1000 domaini kaydet
df_tr.head(1000).to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ Tranco listesinden Türkiye verisi kaydedildi: {output_file}")


📦 Tranco listesi indiriliyor...
✅ Tranco listesinden Türkiye verisi kaydedildi: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top_1000_turkey_tranco.csv


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time, pandas as pd, os

save_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata"
os.makedirs(save_path, exist_ok=True)
output_file = os.path.join(save_path, "top_1000_turkey_cloudflare.csv")

print("🚀 Cloudflare Radar Türkiye verisi çekiliyor...")

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://radar.cloudflare.com/turkey")

time.sleep(8)  # Sayfa yüklenene kadar bekle

# Domain listesi elementlerini bul (Cloudflare DOM yapısı değişebilir)
elems = driver.find_elements(By.CSS_SELECTOR, "a[href^='/domain/']")

domains = []
for e in elems:
    text = e.text.strip()
    if text and "." in text:
        domains.append(text.lower())

driver.quit()

domains = list(dict.fromkeys(domains))[:1000]
pd.DataFrame({"rank": range(1, len(domains)+1), "domain": domains}).to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ {len(domains)} domain bulundu ve kaydedildi: {output_file}")


🚀 Cloudflare Radar Türkiye verisi çekiliyor...
✅ 0 domain bulundu ve kaydedildi: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top_1000_turkey_cloudflare.csv


In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import os

# --- Ayarlar ---
SAVE_PATH = r"C:\\Users\\elifo\\Bitirme_Projesi_Dataset_Olusturma\\Beningdata"
os.makedirs(SAVE_PATH, exist_ok=True)
OUTPUT_CSV = os.path.join(SAVE_PATH, "similarweb_turkey_300.csv")

# --- Tarayıcı ayarları ---
options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,3000")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# Giriş yapabilmek için headless KAPALI olsun:
# (giriş ekranı görünecek)
# options.add_argument("--headless")

# --- Tarayıcıyı başlat ---
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    print("\n🔐 SimilarWeb sayfası açılıyor. Giriş yapmanız gerekebilir...")
    driver.get("https://www.similarweb.com/top-websites/turkey/")
    time.sleep(30)  # Giriş yapmanız için süre tanıyoruz

    print("\n⬇️ Sayfa kaydırılıyor, domainler toplanıyor...")
    domains = []
    scroll_pause = 2
    scroll_count = 0
    max_scroll = 50

    while len(domains) < 300 and scroll_count < max_scroll:
        # Sayfayı en alta kaydır
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause)

        # Domainleri bul
        elems = driver.find_elements(By.CSS_SELECTOR, "a.top-site-list__title, a[data-test='sp_topSiteLink']")
        for e in elems:
            domain = e.text.strip()
            if domain and "." in domain and domain not in domains:
                domains.append(domain)

        scroll_count += 1
        print(f"{len(domains)} domain bulundu...")

    # İlk 300'ü al
    domains = domains[:300]

    # CSV'ye yaz
    df = pd.DataFrame({"rank": range(1, len(domains)+1), "domain": domains})
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ {len(domains)} domain kaydedildi: {OUTPUT_CSV}")

finally:
    driver.quit()


🔐 SimilarWeb sayfası açılıyor. Giriş yapmanız gerekebilir...

⬇️ Sayfa kaydırılıyor, domainler toplanıyor...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain bulundu...
0 domain b

In [3]:
pip install selenium pandas


Note: you may need to restart the kernel to use updated packages.


In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--start-maximized")

service = Service(r"C:\Users\elifo\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe")

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.google.com")
print("✅ ChromeDriver başarıyla çalıştı:", driver.title)
driver.quit()


✅ ChromeDriver başarıyla çalıştı: Google


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

options = Options()
options.add_argument("--start-maximized")

service = Service(r"C:\Users\elifo\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe")
driver = webdriver.Chrome(service=service, options=options)

# 1️⃣ Login ekranı
login_url = "https://pro.similarweb.com/"
driver.get(login_url)
print("🌐 Lütfen giriş yap (email + şifre).")
time.sleep(60)

# 2️⃣ Hedef sayfaya git
target_url = "https://pro.similarweb.com/#/digitalsuite/markets/webmarketanalysis/mapping/All/792/1m?webSource=Total"
driver.get(target_url)
print("🔁 Login tamamlandı, hedef sayfaya gidildi...")
time.sleep(15)

# 3️⃣ Scroll + her adımda domain topla
domains = set()  # tekrarlamaları önlemek için set kullanıyoruz

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_count = 0

while True:
    # Şu anda görünen domainleri al
    elements = driver.find_elements(By.CSS_SELECTOR, "a.ExternalLink-ggolzm.gzGCUk.link-out")
    for el in elements:
        href = el.get_attribute("href")
        if href and href.startswith("http"):
            domain = href.replace("http://", "").replace("https://", "").split("/")[0]
            domains.add(domain)

    # Aşağı kaydır
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    new_height = driver.execute_script("return document.body.scrollHeight")
    scroll_count += 1
    print(f"⬇️ Kaydırma sayısı: {scroll_count}, şu ana kadar {len(domains)} domain toplandı")

    # Yeni veri yüklenmediyse döngüyü bitir
    if new_height == last_height:
        break
    last_height = new_height

print("✅ Scroll işlemi tamamlandı.")
print(f"💾 Toplam {len(domains)} domain bulundu, CSV'ye kaydediliyor...")

# 4️⃣ CSV'ye kaydet
df = pd.DataFrame({"domain": sorted(list(domains))})
df.to_csv("similarweb_domains.csv", index=False, encoding="utf-8-sig")
print("🎉 Kaydedildi -> similarweb_domains.csv")

driver.quit()


🌐 Lütfen giriş yap (email + şifre).
🔁 Login tamamlandı, hedef sayfaya gidildi...
⬇️ Kaydırma sayısı: 1, şu ana kadar 100 domain toplandı
✅ Scroll işlemi tamamlandı.
💾 Toplam 100 domain bulundu, CSV'ye kaydediliyor...
🎉 Kaydedildi -> similarweb_domains.csv


In [1]:
pip install selenium-wire


  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached pyasn1-0.6.1-py3-none-any.whl (83 kB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ----------- ---------------------------- 1.0/3.5 MB 6.3 MB/s eta 0:00:01
   -------------------------- ------------- 2.4/3.5 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 6.3 MB/s  0:00:00

   ----------------------------------------  0/12 [pydivert]
   ---------- -----------------------------  3/12 [pyasn1]
   ---------- -----------------------------  3/12 [pyasn1]
   ------------- --------------------------  4/12 [kaitaistruct]
   -------------------- -------------------  6/12 [hpack]
   -------------------------- -------------  8/12 [h2]
   -------------------------- -------------  8/12 [h2]
   ------------------------------ ---------  9/12 [cryptograph

In [6]:
pip install blinker==1.6.2


  Attempting uninstall: blinker
    Found existing installation: blinker 1.9.0
    Uninstalling blinker-1.9.0:
      Successfully uninstalled blinker-1.9.0
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd

# 1. bp.txt dosyasını oku
# Eğer dosyada sütunlar virgülle, tab ile veya boşlukla ayrılmışsa ona göre ayırıcıyı değiştir
# Örnek: sep="\t" -> tab, sep="," -> virgül, sep=" " -> boşluk
df = pd.read_csv("bp.txt", sep="\t", header=None, encoding="utf-8")

# 2. İlk birkaç satıra bakalım
print("İlk 5 satır:")
print(df.head())

# 3. Tekrarlanan satırları kaldır
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"\n🔹 {before - after} adet tekrarlı satır kaldırıldı. Kalan toplam satır: {after}")

# 4. Yeni dosyayı CSV olarak kaydet
df.to_csv("bp_cleaned.csv", index=False, encoding="utf-8")
print("\n✅ 'bp_cleaned.csv' dosyası başarıyla oluşturuldu.")

# 5. Basit analizler
print("\nVeri Analizi:")
print(df.describe(include="all"))


İlk 5 satır:
             0
0         icon
1   google.com
2         icon
3  youtube.com
4         icon

🔹 9963 adet tekrarlı satır kaldırıldı. Kalan toplam satır: 9966

✅ 'bp_cleaned.csv' dosyası başarıyla oluşturuldu.

Veri Analizi:
                      0
count              9966
unique             9966
top     cloud.microsoft
freq                  1


In [9]:
import pandas as pd

# 1. bp.txt dosyasını oku
# Eğer dosyada sütunlar tab, virgül veya boşluk ile ayrılmışsa sep parametresini değiştir
df = pd.read_csv("bp.txt", sep="\t", header=None, encoding="utf-8", names=["deger"])

# 2. Toplam satır sayısı
print(f"Toplam satır sayısı: {len(df)}")

# 3. Tekrarlanan satırları bul
duplicates = df[df.duplicated(keep=False)]  # tüm tekrar edenleri gösterir
unique_count = len(df.drop_duplicates())

print(f"Tekrarlanan satır sayısı: {len(duplicates)}")
print(f"Eşsiz satır sayısı: {unique_count}\n")

# 4. Tekrarlanan satırları ayrı dosyaya kaydet
if len(duplicates) > 0:
    duplicates.to_csv("bp_duplicates.csv", index=False, encoding="utf-8")
    print("⚠️ Tekrarlanan satırlar 'bp_duplicates.csv' dosyasına kaydedildi.")
else:
    print("✅ Hiç tekrarlanan satır yok.")

# 5. Tekrarsız (unique) veriyi kaydet
df_unique = df.drop_duplicates()
df_unique.to_csv("bp_cleaned.csv", index=False, encoding="utf-8")
print("✅ Tekrarsız veriler 'bp_cleaned.csv' dosyasına kaydedildi.")

# 6. Basit analiz
print("\n📊 Veri Analizi:")
print(df_unique.describe(include="all"))

# 7. İsteğe bağlı olarak ek bilgi: en çok tekrar eden değerler
print("\n🔁 En çok tekrar eden ilk 10 ifade:")
print(df['deger'].value_counts().head(10))


Toplam satır sayısı: 19929
Tekrarlanan satır sayısı: 9964
Eşsiz satır sayısı: 9966

⚠️ Tekrarlanan satırlar 'bp_duplicates.csv' dosyasına kaydedildi.
✅ Tekrarsız veriler 'bp_cleaned.csv' dosyasına kaydedildi.

📊 Veri Analizi:
                  deger
count              9966
unique             9966
top     cloud.microsoft
freq                  1

🔁 En çok tekrar eden ilk 10 ifade:
deger
icon                9964
thepiratebay3.to       1
onedio.com             1
isa-sari.com           1
marksist.org           1
akakce.com             1
cloud.microsoft        1
google.com             1
youtube.com            1
instagram.com          1
Name: count, dtype: int64


In [10]:
import pandas as pd

# CSV'yi oku
df = pd.read_csv("bp_cleaned.csv", encoding="utf-8")

# 1. satırı (index 0) sil
df = df.drop(index=0)

# İstersen indexleri sıfırla
df = df.reset_index(drop=True)

# Güncellenmiş dosyayı kaydet
df.to_csv("bp_cleaned.csv", index=False, encoding="utf-8")

print("✅ 'bp_cleaned.csv' dosyasının 1. satırı başarıyla silindi.")


✅ 'bp_cleaned.csv' dosyasının 1. satırı başarıyla silindi.


In [11]:
pip install aiohttp aiofiles pandas beautifulsoup4 lxml tldextract aiodns



  Using cached aiofiles-25.1.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
Using cached aiofiles-25.1.0-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 8.3 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 6.5 MB/s eta 0:00:01
   ------------------------------- -------- 3.1/4.0 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 5.6 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)

   ------ ---------------------------------  2/12 [lxml]
   ------ ---------------------------------  2/12 [lxml]
   ------ 

In [12]:
!pip install aiohttp aiofiles pandas beautifulsoup4 lxml tldextract aiodns tqdm


In [1]:
print(3)

3


In [3]:
import pandas as pd
import requests, time, re
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import tldextract
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ============================
# 🧭 AYARLAR
# ============================
INPUT_CSV = "merged_domain_list.csv"
OUTPUT_GOOD = "good_urls.csv"
OUTPUT_BAD = "donturlstatus.csv"

WORKERS = 100
REQUEST_TIMEOUT = 10
DOMAIN_TIMEOUT = 100
PER_DOMAIN_LIMIT = 15
FIRST_LEVEL_LIMIT = 8
SECOND_LEVEL_LIMIT = 6
RETRIES = 2
BACKOFF = 0.8
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; bp-crawler/3.5)"}
TEST_DOMAIN_COUNT = 500
SAVE_INTERVAL = 100  # her 100 domainde CSV yaz

# ============================
# 🔧 YARDIMCI FONKSİYONLAR
# ============================
def reg_domain(s):
    ex = tldextract.extract(str(s))
    return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None

def fetch(url):
    for attempt in range(RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            return r
        except Exception:
            if attempt < RETRIES:
                time.sleep(BACKOFF * (attempt + 1))
    return None

def status_ok(url):
    r = fetch(url)
    if r is not None:
        return (r.status_code == 200, r.status_code, (r.text if r.ok else None), r.url)
    return (False, None, None, url)

def parse_sitemap(text):
    urls = re.findall(r"<loc>(.*?)</loc>", text, flags=re.IGNORECASE | re.DOTALL)
    cleaned = [u.strip().split("#")[0].rstrip("/") for u in urls if u.strip()]
    return list(dict.fromkeys(cleaned))

def extract_internal_links(base_url, html, domain):
    soup = BeautifulSoup(html, "lxml")
    found = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "tel:", "javascript:")):
            continue
        try:
            abs_url = urljoin(base_url, href)
            p = urlparse(abs_url)
            if p.netloc and p.netloc.endswith(domain):
                found.add(abs_url.split("#")[0].rstrip("/"))
        except Exception:
            continue
    try:
        c = soup.find("link", rel=lambda x: x and "canonical" in x)
        if c and c.get("href"):
            found.add(urljoin(base_url, c["href"]).split("#")[0].rstrip("/"))
    except Exception:
        pass
    found.add(base_url.rstrip("/"))
    return list(dict.fromkeys(found))

def process_one_domain(domain):
    start = time.time()
    def timed_out(): return (time.time() - start) > DOMAIN_TIMEOUT

    ok_rows = []
    # 1️⃣ Sitemap
    for base in (f"https://{domain}", f"http://{domain}"):
        if timed_out(): break
        r = fetch(base + "/sitemap.xml")
        if r and r.ok and "<loc>" in r.text:
            urls = parse_sitemap(r.text)
            urls = [u for u in urls if urlparse(u).netloc.endswith(domain)]
            for u in urls:
                if timed_out(): break
                is_ok, code, _, _ = status_ok(u)
                if is_ok:
                    ok_rows.append({"domain": domain, "url": u, "status": code})
                    if len(ok_rows) >= PER_DOMAIN_LIMIT:
                        return {"ok": ok_rows, "bad": None}
            break

    # 2️⃣ Ana sayfa + link zinciri
    for base in (f"https://{domain}", f"http://{domain}"):
        if timed_out(): break
        r = fetch(base)
        if not (r and r.ok and r.text):
            continue
        is_ok, code, html, final_url = status_ok(r.url)
        if is_ok:
            ok_rows.append({"domain": domain, "url": final_url.rstrip("/"), "status": code})
            if len(ok_rows) >= PER_DOMAIN_LIMIT:
                return {"ok": ok_rows, "bad": None}

        first_links = extract_internal_links(r.url, r.text, domain)[:FIRST_LEVEL_LIMIT]
        for link in first_links:
            if timed_out(): break
            is_ok, code, html, final = status_ok(link)
            if is_ok:
                ok_rows.append({"domain": domain, "url": final.rstrip("/"), "status": code})
                if len(ok_rows) >= PER_DOMAIN_LIMIT:
                    return {"ok": ok_rows, "bad": None}

                if html and not timed_out():
                    second_links = extract_internal_links(final, html, domain)[:SECOND_LEVEL_LIMIT]
                    for s in second_links:
                        if timed_out(): break
                        is_ok2, code2, _, final2 = status_ok(s)
                        if is_ok2:
                            ok_rows.append({"domain": domain, "url": final2.rstrip("/"), "status": code2})
                            if len(ok_rows) >= PER_DOMAIN_LIMIT:
                                return {"ok": ok_rows, "bad": None}
        break

    if len(ok_rows) == 0:
        return {"ok": [], "bad": {"domain": domain, "reason": "no_successful_url"}}
    return {"ok": ok_rows, "bad": None}

# ============================
# 🚀 ANA ÇALIŞMA
# ============================
df_domains = pd.read_csv(INPUT_CSV)
col = df_domains.columns[0]
domains = [reg_domain(x) for x in df_domains[col] if pd.notna(x)]
domains = [d for d in domains if d]
domains = list(dict.fromkeys(domains))[:TEST_DOMAIN_COUNT]
print(f"Deneme için işlenecek domain sayısı: {len(domains)}")

good_rows, bad_rows = [], []
batch_no = 1

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(process_one_domain, d): d for d in domains}
    completed = 0
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Domainler (ilk 1000) taranıyor..."):
        try:
            res = fut.result()
            if res["ok"]:
                good_rows.extend(res["ok"])
            if res["bad"]:
                bad_rows.append(res["bad"])
        except Exception:
            bad_rows.append({"domain": futures[fut], "reason": "exception"})

        completed += 1
        if completed % SAVE_INTERVAL == 0:
            pd.DataFrame(good_rows).drop_duplicates(subset=["url"]).to_csv(f"good_urls_partial_{batch_no}.csv", index=False)
            pd.DataFrame(bad_rows).to_csv(f"donturlstatus_partial_{batch_no}.csv", index=False)
            print(f"\n💾 {batch_no}. ara kayıt yapıldı ({completed} domain tamamlandı)")
            batch_no += 1

# ============================
# 💾 FİNAL KAYIT + RAPOR
# ============================
good_df = pd.DataFrame(good_rows).drop_duplicates(subset=["url"]).reset_index(drop=True)
bad_df = pd.DataFrame(bad_rows)
good_df.to_csv(OUTPUT_GOOD, index=False, encoding="utf-8")
bad_df.to_csv(OUTPUT_BAD, index=False, encoding="utf-8")

print("\n📊 Deneme Tarama Özeti")
print("-----------------------------")
print(f"Toplam domain (test): {len(domains)}")
print(f"200 OK üreten domain sayısı: {good_df['domain'].nunique() if not good_df.empty else 0}")
print(f"Hiç başarılı URL çıkmayan/erken kesilen domain sayısı: {len(bad_df)}")
print(f"Toplam 200 OK URL: {len(good_df)}")
print("-----------------------------")
print(f"✅ {OUTPUT_GOOD}  (final başarılı URL'ler)")
print(f"⚠️ {OUTPUT_BAD}   (final başarısız/timeout/erişilemeyen)")


C:\Users\elifo\AppData\Local\Temp\ipykernel_2012\904695584.py:35: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None


Deneme için işlenecek domain sayısı: 500


Domainler (ilk 1000) taranıyor...:  20%|█████████                                    | 101/500 [01:41<01:25,  4.69it/s]


💾 1. ara kayıt yapıldı (100 domain tamamlandı)


Domainler (ilk 1000) taranıyor...:  40%|██████████████████                           | 200/500 [02:34<05:47,  1.16s/it]

...


💾 3. ara kayıt yapıldı (300 domain tamamlandı)


Domainler (ilk 1000) taranıyor...:  81%|████████████████████████████████████▎        | 403/500 [05:13<00:58,  1.66it/s]


💾 4. ara kayıt yapıldı (400 domain tamamlandı)


Domainler (ilk 1000) taranıyor...: 100%|█████████████████████████████████████████████| 500/500 [09:56<00:00,  1.19s/it]


💾 5. ara kayıt yapıldı (500 domain tamamlandı)

📊 Deneme Tarama Özeti
-----------------------------
Toplam domain (test): 500
200 OK üreten domain sayısı: 257
Hiç başarılı URL çıkmayan/erken kesilen domain sayısı: 232
Toplam 200 OK URL: 1389
-----------------------------
✅ good_urls.csv  (final başarılı URL'ler)
⚠️ donturlstatus.csv   (final başarısız/timeout/erişilemeyen)


In [4]:
# !pip install -q pandas requests beautifulsoup4 lxml tqdm tldextract

import pandas as pd
import requests, time, re, socket
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import tldextract
import warnings

# Uyarıları gizle
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# ============================
# ⚙️ AYARLAR
# ============================
INPUT_CSV = "merged_domain_list.csv"
OUTPUT_GOOD = "good_urls.csv"
OUTPUT_BAD = "donturlstatus.csv"

WORKERS = 100
REQUEST_TIMEOUT = 12          # tek istek süresi
DOMAIN_TIMEOUT = 45           # domain başına toplam süre
PER_DOMAIN_LIMIT = 25         # en fazla kaç URL kaydedilsin
FIRST_LEVEL_LIMIT = 10
SECOND_LEVEL_LIMIT = 8
RETRIES = 3
BACKOFF = 1.0
SAVE_INTERVAL = 100           # her 100 domainde CSV yaz
TEST_DOMAIN_COUNT = 500

# Tarayıcı benzeri header
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
}

# ============================
# 🧩 YARDIMCI FONKSİYONLAR
# ============================
def reg_domain(s):
    ex = tldextract.extract(str(s))
    return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None

def dns_resolves(domain):
    try:
        socket.gethostbyname(domain)
        return True
    except Exception:
        return False

def fetch(url):
    for attempt in range(RETRIES + 1):
        try:
            r = requests.get(
                url,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT,
                allow_redirects=True,
                verify=False,
            )
            return r
        except Exception:
            if attempt < RETRIES:
                time.sleep(BACKOFF * (attempt + 1))
    return None

def status_ok(url):
    r = fetch(url)
    if r is not None:
        return (r.status_code == 200, r.status_code, (r.text if r.ok else None), r.url)
    return (False, None, None, url)

def parse_sitemap(text):
    urls = re.findall(r"<loc>(.*?)</loc>", text, flags=re.IGNORECASE | re.DOTALL)
    cleaned = [u.strip().split("#")[0].rstrip("/") for u in urls if u.strip()]
    return list(dict.fromkeys(cleaned))

def extract_internal_links(base_url, html, domain):
    soup = BeautifulSoup(html, "lxml")
    found = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "tel:", "javascript:")):
            continue
        try:
            abs_url = urljoin(base_url, href)
            p = urlparse(abs_url)
            if p.netloc and p.netloc.endswith(domain):
                found.add(abs_url.split("#")[0].rstrip("/"))
        except Exception:
            continue
    try:
        c = soup.find("link", rel=lambda x: x and "canonical" in x)
        if c and c.get("href"):
            found.add(urljoin(base_url, c["href"]).split("#")[0].rstrip("/"))
    except Exception:
        pass
    found.add(base_url.rstrip("/"))
    return list(dict.fromkeys(found))

def find_sitemap_in_robots(domain):
    """robots.txt içinden sitemap URL’ini çekmeye çalışır"""
    for base in (f"https://{domain}", f"http://{domain}"):
        r = fetch(base + "/robots.txt")
        if r and r.ok and "Sitemap:" in r.text:
            matches = re.findall(r"Sitemap:\s*(\S+)", r.text, re.IGNORECASE)
            if matches:
                return matches
    return []

# ============================
# 🔁 DOMAIN TARAYICI
# ============================
def process_one_domain(domain):
    start = time.time()
    def timed_out(): return (time.time() - start) > DOMAIN_TIMEOUT
    ok_rows = []

    # 1️⃣ DNS çözülüyor mu?
    if not dns_resolves(domain):
        return {"ok": [], "bad": {"domain": domain, "reason": "dns_failed"}}

    # 2️⃣ robots.txt içinde sitemap var mı?
    robots_maps = find_sitemap_in_robots(domain)
    sitemap_candidates = robots_maps + [f"https://{domain}/sitemap.xml", f"http://{domain}/sitemap.xml"]

    # 3️⃣ sitemap’leri sırayla dene
    for sm in sitemap_candidates:
        if timed_out(): break
        r = fetch(sm)
        if r and r.ok and "<loc>" in r.text:
            urls = parse_sitemap(r.text)
            urls = [u for u in urls if urlparse(u).netloc.endswith(domain)]
            for u in urls:
                if timed_out(): break
                is_ok, code, _, _ = status_ok(u)
                if is_ok:
                    ok_rows.append({"domain": domain, "url": u, "status": code})
                    if len(ok_rows) >= PER_DOMAIN_LIMIT:
                        return {"ok": ok_rows, "bad": None}
            break  # sitemap bulunduysa yeter

    # 4️⃣ ana sayfa varyasyonları (www / non-www)
    bases = [
        f"https://{domain}",
        f"http://{domain}",
        f"https://www.{domain}",
        f"http://www.{domain}",
    ]
    for base in bases:
        if timed_out(): break
        r = fetch(base)
        if not (r and r.ok and r.text):
            continue
        is_ok, code, html, final_url = status_ok(r.url)
        if is_ok:
            ok_rows.append({"domain": domain, "url": final_url.rstrip("/"), "status": code})
            if len(ok_rows) >= PER_DOMAIN_LIMIT:
                return {"ok": ok_rows, "bad": None}

        first_links = extract_internal_links(r.url, r.text, domain)[:FIRST_LEVEL_LIMIT]
        for link in first_links:
            if timed_out(): break
            is_ok, code, html, final = status_ok(link)
            if is_ok:
                ok_rows.append({"domain": domain, "url": final.rstrip("/"), "status": code})
                if len(ok_rows) >= PER_DOMAIN_LIMIT:
                    return {"ok": ok_rows, "bad": None}

                if html and not timed_out():
                    second_links = extract_internal_links(final, html, domain)[:SECOND_LEVEL_LIMIT]
                    for s in second_links:
                        if timed_out(): break
                        is_ok2, code2, _, final2 = status_ok(s)
                        if is_ok2:
                            ok_rows.append({"domain": domain, "url": final2.rstrip("/"), "status": code2})
                            if len(ok_rows) >= PER_DOMAIN_LIMIT:
                                return {"ok": ok_rows, "bad": None}
        break

    if len(ok_rows) == 0:
        return {"ok": [], "bad": {"domain": domain, "reason": "no_successful_url"}}
    return {"ok": ok_rows, "bad": None}

# ============================
# 🚀 ANA ÇALIŞMA (1000 DOMAIN)
# ============================
df_domains = pd.read_csv(INPUT_CSV)
col = df_domains.columns[0]
domains = [reg_domain(x) for x in df_domains[col] if pd.notna(x)]
domains = [d for d in domains if d]
domains = list(dict.fromkeys(domains))[:TEST_DOMAIN_COUNT]
print(f"Deneme için işlenecek domain sayısı: {len(domains)}")

good_rows, bad_rows = [], []
batch_no = 1

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(process_one_domain, d): d for d in domains}
    completed = 0
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Domainler taranıyor..."):
        try:
            res = fut.result()
            if res["ok"]:
                good_rows.extend(res["ok"])
            if res["bad"]:
                bad_rows.append(res["bad"])
        except Exception:
            bad_rows.append({"domain": futures[fut], "reason": "exception"})

        completed += 1
        if completed % SAVE_INTERVAL == 0:
            pd.DataFrame(good_rows).drop_duplicates(subset=["url"]).to_csv(f"good_urls_partial_{batch_no}.csv", index=False)
            pd.DataFrame(bad_rows).to_csv(f"donturlstatus_partial_{batch_no}.csv", index=False)
            print(f"\n💾 {batch_no}. ara kayıt yapıldı ({completed} domain tamamlandı)")
            batch_no += 1

# ============================
# 💾 FİNAL KAYIT + RAPOR
# ============================
good_df = pd.DataFrame(good_rows).drop_duplicates(subset=["url"]).reset_index(drop=True)
bad_df = pd.DataFrame(bad_rows)

good_df.to_csv(OUTPUT_GOOD, index=False, encoding="utf-8")
bad_df.to_csv(OUTPUT_BAD, index=False, encoding="utf-8")

print("\n📊 Tarama Özeti")
print("-----------------------------")
print(f"Toplam domain (test): {len(domains)}")
print(f"200 OK üreten domain sayısı: {good_df['domain'].nunique() if not good_df.empty else 0}")
print(f"Hiç başarılı URL çıkmayan domain sayısı: {len(bad_df)}")
print(f"Toplam 200 OK URL: {len(good_df)}")
print("-----------------------------")
print(f"✅ {OUTPUT_GOOD}  (final başarılı URL'ler)")
print(f"⚠️ {OUTPUT_BAD}   (final başarısız/timeout/erişilemeyen)")


C:\Users\elifo\AppData\Local\Temp\ipykernel_2012\3894488845.py:48: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None


Deneme için işlenecek domain sayısı: 500


Domainler taranıyor...:  20%|███████████▎                                            | 101/500 [01:44<06:30,  1.02it/s]


💾 1. ara kayıt yapıldı (100 domain tamamlandı)


Domainler taranıyor...:  40%|██████████████████████▍                                 | 200/500 [02:16<00:11, 25.30it/s]

...


💾 3. ara kayıt yapıldı (300 domain tamamlandı)


Domainler taranıyor...:  80%|████████████████████████████████████████████▉           | 401/500 [03:57<00:56,  1.74it/s]


💾 4. ara kayıt yapıldı (400 domain tamamlandı)


Domainler taranıyor...: 100%|████████████████████████████████████████████████████████| 500/500 [11:33<00:00,  1.39s/it]


💾 5. ara kayıt yapıldı (500 domain tamamlandı)

📊 Tarama Özeti
-----------------------------
Toplam domain (test): 500
200 OK üreten domain sayısı: 245
Hiç başarılı URL çıkmayan domain sayısı: 248
Toplam 200 OK URL: 868
-----------------------------
✅ good_urls.csv  (final başarılı URL'ler)
⚠️ donturlstatus.csv   (final başarısız/timeout/erişilemeyen)


In [5]:
# !pip install -q pandas requests beautifulsoup4 lxml tqdm tldextract

import pandas as pd
import requests, time, re, socket
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import tldextract, warnings
from collections import deque
from pathlib import Path

# Uyarıları gizle
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# ============================
# ⚙️ AYARLAR (büyük veri için optimize)
# ============================
INPUT_CSV = "merged_domain_list.csv"
OUTPUT_GOOD = "good_urls.csv"
OUTPUT_BAD = "donturlstatus.csv"

WORKERS = 60                 # -> 60 thread
REQUEST_TIMEOUT = 12         # tek istek süresi (s)
DOMAIN_TIMEOUT = 60          # domain başına toplam çalışma süresi (s) – içsel kontrol
PER_DOMAIN_LIMIT = 20       # bir domain başına hedef en fazla başarılı URL
CRAWL_DEPTH = 3              # BFS derinliği (1..3)
MAX_CRAWL_QUEUE = 800        # bir domain içinde en fazla kuyruğa alınacak URL
RETRIES = 3
BACKOFF = 1.0
POLITENESS_DELAY = 0.10      # her istek sonrası küçük gecikme (s)
SAVE_INTERVAL = 5000         # -> her 5000 domainde ara kayıt
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
}

# ============================
# 🔧 YARDIMCI FONKSİYONLAR
# ============================
def reg_domain(s):
    ex = tldextract.extract(str(s))
    # yeni isim (varsa) yoksa eskiye düş
    return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None

def dns_resolves(domain):
    try:
        socket.gethostbyname(domain)
        return True
    except Exception:
        return False

def fetch(url):
    """Retry + backoff + verify=False + küçük gecikme"""
    for attempt in range(RETRIES + 1):
        try:
            r = requests.get(
                url,
                headers=HEADERS,
                timeout=REQUEST_TIMEOUT,
                allow_redirects=True,
                verify=False,
            )
            time.sleep(POLITENESS_DELAY)
            return r
        except Exception:
            if attempt < RETRIES:
                time.sleep(BACKOFF * (attempt + 1))
    return None

def parse_sitemap(text):
    urls = re.findall(r"<loc>(.*?)</loc>", text, flags=re.IGNORECASE | re.DOTALL)
    cleaned = [u.strip().split("#")[0].rstrip("/") for u in urls if u.strip()]
    return list(dict.fromkeys(cleaned))

def extract_internal_links(base_url, html, domain):
    soup = BeautifulSoup(html, "lxml")
    found = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "tel:", "javascript:")):
            continue
        try:
            abs_url = urljoin(base_url, href)
            p = urlparse(abs_url)
            if p.scheme in ("http","https") and p.netloc and p.netloc.endswith(domain):
                found.add(abs_url.split("#")[0].rstrip("/"))
        except Exception:
            continue
    # canonical
    try:
        c = soup.find("link", rel=lambda x: x and "canonical" in x)
        if c and c.get("href"):
            found.add(urljoin(base_url, c["href"]).split("#")[0].rstrip("/"))
    except Exception:
        pass
    # homepage
    found.add(base_url.rstrip("/"))
    return list(dict.fromkeys(found))

def find_sitemap_in_robots(domain):
    """robots.txt içinden Sitemap satırlarını bul"""
    candidates = []
    for base in (f"https://{domain}", f"http://{domain}", f"https://www.{domain}", f"http://www.{domain}"):
        r = fetch(base + "/robots.txt")
        if r and r.ok and "Sitemap:" in r.text:
            matches = re.findall(r"Sitemap:\s*(\S+)", r.text, re.IGNORECASE)
            candidates.extend(matches)
    return candidates

# ============================
# 🔁 GELİŞMİŞ CRAWLER (BFS + sitemap index + içsel timeout)
# ============================
def process_one_domain(domain):
    start = time.time()
    def timed_out(): return (time.time() - start) > DOMAIN_TIMEOUT

    # DNS kontrol
    if not dns_resolves(domain):
        return {"ok": [], "bad": {"domain": domain, "reason": "dns_failed"}}

    good = []
    visited = set()
    queue = deque()

    # 1) robots.txt → sitemap adayları
    sitemap_candidates = find_sitemap_in_robots(domain)
    # 2) tipik sitemap yolları
    sitemap_candidates += [
        f"https://{domain}/sitemap.xml", f"http://{domain}/sitemap.xml",
        f"https://www.{domain}/sitemap.xml", f"http://www.{domain}/sitemap.xml"
    ]

    # 3) sitemap'leri dene (sitemap index desteği)
    for sm in sitemap_candidates:
        if timed_out(): break
        r = fetch(sm)
        if not (r and r.ok and r.text):
            continue
        # sitemap index (başka sitemap listeleri)
        inner = re.findall(r"<sitemap>\s*<loc>(.*?)</loc>", r.text, flags=re.IGNORECASE|re.DOTALL)
        if inner:
            for i in inner:
                if timed_out(): break
                if urlparse(i).netloc.endswith(domain):
                    r2 = fetch(i)
                    if r2 and r2.ok and r2.text:
                        for u in parse_sitemap(r2.text):
                            if urlparse(u).netloc.endswith(domain):
                                if u not in visited:
                                    visited.add(u)
                                    queue.append((u, 1))
                            if len(visited) >= MAX_CRAWL_QUEUE:
                                break
        else:
            # düz sitemap
            for u in parse_sitemap(r.text):
                if urlparse(u).netloc.endswith(domain):
                    if u not in visited:
                        visited.add(u)
                        queue.append((u, 1))
                if len(visited) >= MAX_CRAWL_QUEUE:
                    break
        if len(visited) >= MAX_CRAWL_QUEUE or timed_out():
            break

    # 4) sitemap yoksa, ana sayfa varyasyonları
    if not queue and not timed_out():
        for b in (f"https://{domain}", f"http://{domain}", f"https://www.{domain}", f"http://www.{domain}"):
            if timed_out(): break
            r = fetch(b)
            if r and r.ok and r.text:
                final = r.url.rstrip("/")
                if final not in visited:
                    visited.add(final)
                    queue.append((final, 1))
                break

    # 5) BFS tarama
    while queue and not timed_out() and len(good) < PER_DOMAIN_LIMIT:
        url, depth = queue.popleft()
        r = fetch(url)
        if not r:
            continue
        if r.status_code == 200:
            # URL uniq kontrolü
            if url.rstrip("/") not in {g["url"] for g in good}:
                good.append({"domain": domain, "url": url.rstrip("/"), "status": r.status_code})

        # HTML ise ve derinlik uygunsa
        content_type = r.headers.get("Content-Type", "")
        if "html" in content_type and depth < CRAWL_DEPTH:
            try:
                links = extract_internal_links(r.url, r.text, domain)
            except Exception:
                links = []
            for L in links:
                if len(visited) >= MAX_CRAWL_QUEUE or timed_out():
                    break
                if L not in visited:
                    visited.add(L)
                    queue.append((L, depth + 1))

    if not good:
        return {"ok": [], "bad": {"domain": domain, "reason": "no_successful_url_or_timeout"}}
    return {"ok": good[:PER_DOMAIN_LIMIT], "bad": None}

# ============================
# 🚀 ANA ÇALIŞMA (5K CHECKPOINT)
# ============================
# Domainleri oku ve uniq hale getir
df_domains = pd.read_csv(INPUT_CSV)
col = df_domains.columns[0]
domains = [reg_domain(x) for x in df_domains[col] if pd.notna(x)]
domains = [d for d in domains if d]
domains = list(dict.fromkeys(domains))
print(f"İşlenecek domain sayısı: {len(domains)}")

# Çıktı klasörü
Path(".").mkdir(parents=True, exist_ok=True)

good_rows, bad_rows = [], []
batch_no = 1
completed = 0
part_good_files, part_bad_files = [], []

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(process_one_domain, d): d for d in domains}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Domainler (büyük tarama) taranıyor..."):
        d = futures[fut]
        try:
            res = fut.result()
            if res["ok"]:
                good_rows.extend(res["ok"])
            if res["bad"]:
                bad_rows.append(res["bad"])
        except Exception:
            bad_rows.append({"domain": d, "reason": "exception"})

        completed += 1
        # 5K CHECKPOINT
        if completed % SAVE_INTERVAL == 0:
            good_part = pd.DataFrame(good_rows)
            bad_part = pd.DataFrame(bad_rows)
            # uniq URL
            if not good_part.empty:
                good_part["url"] = good_part["url"].astype(str).str.strip().str.rstrip("/")
                good_part = good_part.drop_duplicates(subset=["url"]).reset_index(drop=True)

            gp = f"good_urls_part_{batch_no}.csv"
            bp = f"donturlstatus_part_{batch_no}.csv"
            good_part.to_csv(gp, index=False, encoding="utf-8")
            bad_part.to_csv(bp, index=False, encoding="utf-8")
            part_good_files.append(gp)
            part_bad_files.append(bp)

            print(f"\n💾 {batch_no}. ara kayıt yapıldı ({completed} domain tamamlandı) → {gp}, {bp}")

            # RAM'i boşalt
            good_rows.clear()
            bad_rows.clear()
            batch_no += 1

# ——— Son kalanları da part olarak yaz
if good_rows or bad_rows:
    good_part = pd.DataFrame(good_rows)
    bad_part = pd.DataFrame(bad_rows)
    if not good_part.empty:
        good_part["url"] = good_part["url"].astype(str).str.strip().str.rstrip("/")
        good_part = good_part.drop_duplicates(subset=["url"]).reset_index(drop=True)
    gp = f"good_urls_part_{batch_no}.csv"
    bp = f"donturlstatus_part_{batch_no}.csv"
    good_part.to_csv(gp, index=False, encoding="utf-8")
    bad_part.to_csv(bp, index=False, encoding="utf-8")
    part_good_files.append(gp)
    part_bad_files.append(bp)
    print(f"\n💾 {batch_no}. son ara kayıt yazıldı ({completed} domain tamamlandı) → {gp}, {bp}")

# ============================
# 📦 PART DOSYALARINI BİRLEŞTİR → FİNAL
# ============================
# GOOD
final_good = []
for p in part_good_files:
    try:
        final_good.append(pd.read_csv(p))
    except Exception:
        pass
if final_good:
    good_df = pd.concat(final_good, ignore_index=True)
    good_df["url"] = good_df["url"].astype(str).str.strip().str.rstrip("/")
    good_df = good_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
else:
    good_df = pd.DataFrame(columns=["domain","url","status"])
good_df.to_csv(OUTPUT_GOOD, index=False, encoding="utf-8")

# BAD
final_bad = []
for p in part_bad_files:
    try:
        final_bad.append(pd.read_csv(p))
    except Exception:
        pass
if final_bad:
    bad_df = pd.concat(final_bad, ignore_index=True)
    bad_df = bad_df.drop_duplicates().reset_index(drop=True)
else:
    bad_df = pd.DataFrame(columns=["domain","reason"])
bad_df.to_csv(OUTPUT_BAD, index=False, encoding="utf-8")

# ============================
# 📊 ÖZET
# ============================
print("\n📊 Büyük Tarama Özeti")
print("-----------------------------")
print(f"Toplam domain: {len(domains)}")
print(f"200 OK üreten domain sayısı: {good_df['domain'].nunique() if not good_df.empty else 0}")
print(f"Hiç başarılı URL çıkmayan domain sayısı: {len(bad_df)}")
print(f"Toplam 200 OK URL: {len(good_df)}")
print("-----------------------------")
print(f"✅ {OUTPUT_GOOD}  (final başarılı URL'ler)")
print(f"⚠️ {OUTPUT_BAD}   (final başarısız/timeout/erişilemeyen)")
print(f"🧩 Ara kayıt parçaları: {len(part_good_files)} good, {len(part_bad_files)} bad")


C:\Users\elifo\AppData\Local\Temp\ipykernel_2012\1770912425.py:48: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return getattr(ex, "top_domain_under_public_suffix", None) or ex.registered_domain or None


İşlenecek domain sayısı: 32598


Domainler (büyük tarama) taranıyor...:  15%|█████▏                            | 5001/32598 [1:29:39<6:20:03,  1.21it/s]


💾 1. ara kayıt yapıldı (5000 domain tamamlandı) → good_urls_part_1.csv, donturlstatus_part_1.csv


Domainler (büyük tarama) taranıyor...:  31%|█████████▊                      | 10000/32598 [3:02:50<13:28:04,  2.15s/it]

...


💾 5. ara kayıt yapıldı (25000 domain tamamlandı) → good_urls_part_5.csv, donturlstatus_part_5.csv


Domainler (büyük tarama) taranıyor...:  92%|██████████████████████████████▎  | 30000/32598 [9:25:42<1:41:29,  2.34s/it]


💾 6. ara kayıt yapıldı (30000 domain tamamlandı) → good_urls_part_6.csv, donturlstatus_part_6.csv


Domainler (büyük tarama) taranıyor...: 100%|██████████████████████████████████| 32598/32598 [10:20:37<00:00,  1.14s/it]



💾 7. son ara kayıt yazıldı (32598 domain tamamlandı) → good_urls_part_7.csv, donturlstatus_part_7.csv

📊 Büyük Tarama Özeti
-----------------------------
Toplam domain: 32598
200 OK üreten domain sayısı: 13851
Hiç başarılı URL çıkmayan domain sayısı: 18362
Toplam 200 OK URL: 39793
-----------------------------
✅ good_urls.csv  (final başarılı URL'ler)
⚠️ donturlstatus.csv   (final başarısız/timeout/erişilemeyen)
🧩 Ara kayıt parçaları: 7 good, 7 bad


In [1]:
print(3)

3


In [2]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# --------------------------
# AYARLAR
# --------------------------
NUM_THREADS = 60
TIMEOUT = 3
TEST_LIMIT = 1000

requests.packages.urllib3.disable_warnings()

# --------------------------
# DOMAIN LİSTESİ
# --------------------------
df = pd.read_csv("merged_domain_list.csv", header=None, names=["domain"])
domains = df["domain"].dropna().astype(str).tolist()

if domains[0].lower() == "domain":
    domains = domains[1:]

test_domains = domains[:TEST_LIMIT]

print(f"🚀 Test başlıyor... İlk {TEST_LIMIT} domain işlenecek.\n")


# --------------------------
# URL VARYASYONLARI
# --------------------------
def generate_url_variants(domain):
    base = domain.strip()
    return [
        f"https://{base}/",
        f"https://www.{base}/",
        f"http://{base}/",
        f"http://www.{base}/",
        f"https://{base}",
        f"https://www.{base}",
        f"http://{base}",
        f"http://www.{base}",
        f"https://{base}/index.html",
        f"http://{base}/index.html",
        f"https://www.{base}/index.html",
        f"http://www.{base}/index.html",
    ]


# --------------------------
# ANA URL BULMA
# --------------------------
def find_main_url(domain):
    print(f"→ test ediliyor: {domain}")
    variants = generate_url_variants(domain)

    for url in variants:
        try:
            r = requests.get(url, timeout=TIMEOUT, verify=False, allow_redirects=True)
            if r.status_code < 400:
                print(f"✔ bulundu: {url} ({r.status_code})")
                return domain, url, r.status_code
        except:
            continue

    print(f"✖ bulunamadı: {domain}")
    return domain, None, None


# --------------------------
# MULTI-THREAD İŞLEME
# --------------------------
results = []
count = 0
start_time = time.time()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {executor.submit(find_main_url, d): d for d in test_domains}

    for future in as_completed(futures):
        count += 1

        # %10 ilerleme
        if count % 50 == 0:
            print(f"📌 İlerleme: {count}/{TEST_LIMIT} tamamlandı")

        try:
            domain, url, status = future.result()
            results.append((domain, url, status))
        except Exception:
            results.append((futures[future], None, None))


elapsed = time.time() - start_time
print(f"\n⏳ Test tamamlandı! Geçen süre: {elapsed:.2f} saniye")


# --------------------------
# KAYDET
# --------------------------
output_df = pd.DataFrame(results, columns=["domain", "main_url", "status"])
output_df.to_csv("test_output_first_1000.csv", index=False)

print("\n📁 test_output_first_1000.csv kaydedildi.")
print(output_df.head(20))


🚀 Test başlıyor... İlk 1000 domain işlenecek.

→ test ediliyor: google.com
→ test ediliyor: mail.ru
→ test ediliyor: microsoft.com
→ test ediliyor: facebook.com
→ test ediliyor: cloudflare.com
→ test ediliyor: amazonaws.com
→ test ediliyor: googleapis.com
→ test ediliyor: dzen.ru
→ test ediliyor: youtube.com
→ test ediliyor: apple.com
→ test ediliyor: instagram.com
→ test ediliyor: gstatic.com
→ test ediliyor: office.com
→ test ediliyor: twitter.com
→ test ediliyor: gtld-servers.net
→ test ediliyor: akamai.net
→ test ediliyor: live.com
→ test ediliyor: linkedin.com
→ test ediliyor: googlevideo.com
→ test ediliyor: akamaiedge.net
→ test ediliyor: googletagmanager.com
→ test ediliyor: amazon.com
→ test ediliyor: fbcdn.net
→ test ediliyor: akadns.net
→ test ediliyor: microsoftonline.com
→ test ediliyor: workers.dev
→ test ediliyor: doubleclick.net
→ test ediliyor: googleusercontent.com
→ test ediliyor: windowsupdate.com
→ test ediliyor: wikipedia.org
→ test ediliyor: ax-msedge.net
→ test 

In [3]:
import pandas as pd

df = pd.read_csv("test_output_first_1000.csv")

# Status kodlarının frekansı
status_counts = df["status"].value_counts(dropna=False)

print("📊 Status Kodları Dağılımı:")
print(status_counts)

# Ayrıca kaç tane domain hiç çalışmamış (main_url None)
no_url = df[df["main_url"].isna()].shape[0]
print(f"\n❌ Hiç URL bulunamayan domain sayısı: {no_url}")


📊 Status Kodları Dağılımı:
status
200.0    589
NaN      407
202.0      3
204.0      1
Name: count, dtype: int64

❌ Hiç URL bulunamayan domain sayısı: 407


In [4]:
import pandas as pd
import requests
import socket
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# ---------------------------------------------------------
# AYARLAR
# ---------------------------------------------------------
NUM_THREADS = 60
TIMEOUT = 4  # gelişmiş tarama için biraz daha güvenli
requests.packages.urllib3.disable_warnings()

# ---------------------------------------------------------
# DOSYAYI YÜKLE
# ---------------------------------------------------------
df = pd.read_csv("test_output_first_1000.csv")

# Ana URL veya status NaN olan domainleri seç
targets = df[df["main_url"].isna() | df["status"].isna()]
domains_to_fix = targets["domain"].tolist()

print(f"🎯 Gelişmiş taramaya girecek domain sayısı: {len(domains_to_fix)}\n")


# ---------------------------------------------------------
# DNS KONTROLÜ
# ---------------------------------------------------------
def dns_exists(domain):
    try:
        socket.gethostbyname(domain)
        return True
    except:
        return False


# ---------------------------------------------------------
# URL VARYASYONLARI
# ---------------------------------------------------------
def build_variants(domain):
    return [
        f"https://{domain}",
        f"https://{domain}/",
        f"https://www.{domain}",
        f"https://www.{domain}/",
        f"http://{domain}",
        f"http://{domain}/",
        f"http://www.{domain}",
        f"http://www.{domain}/",
        f"https://{domain}/index.html",
        f"http://{domain}/index.html",
    ]


# ---------------------------------------------------------
# GÜÇLÜ USER-AGENT
# ---------------------------------------------------------
headers = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}


# ---------------------------------------------------------
# GELİŞMİŞ TARAYICI
# ---------------------------------------------------------
def advanced_scan(domain):

    # 1) DNS kontrolü
    if not dns_exists(domain):
        return domain, None, None

    variants = build_variants(domain)

    for url in variants:

        # Önce HEAD dene
        try:
            r = requests.head(url, timeout=TIMEOUT, verify=False, allow_redirects=True,
                              headers=headers)
            if r.status_code < 400:
                # HEAD başarılı → GET ile doğrula
                try:
                    g = requests.get(url, timeout=TIMEOUT, verify=False,
                                     allow_redirects=True, headers=headers)
                    if g.status_code < 400:
                        return domain, url, g.status_code
                except:
                    pass
        except:
            pass

        # HEAD başarısız → direkt GET fallback
        try:
            g = requests.get(url, timeout=TIMEOUT, verify=False,
                             allow_redirects=True, headers=headers)
            if g.status_code < 400:
                return domain, url, g.status_code
        except:
            continue

    return domain, None, None


# ---------------------------------------------------------
# MULTI-THREAD ÇALIŞTIRMA
# ---------------------------------------------------------
results = {}
start_time = time.time()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {executor.submit(advanced_scan, d): d for d in domains_to_fix}

    for future in as_completed(futures):
        domain, url, status = future.result()
        results[domain] = (url, status)


elapsed = time.time() - start_time
print(f"\n⏳ Gelişmiş tarama tamamlandı! Süre: {elapsed:.2f} saniye\n")


# ---------------------------------------------------------
# CSV'Yİ YERİNDE GÜNCELLE
# ---------------------------------------------------------
for domain, (url, status) in results.items():
    df.loc[df["domain"] == domain, "main_url"] = url
    df.loc[df["domain"] == domain, "status"] = status

df.to_csv("test_output_first_1000.csv", index=False)

print("📁 Dosya başarıyla güncellendi!")


🎯 Gelişmiş taramaya girecek domain sayısı: 407


⏳ Gelişmiş tarama tamamlandı! Süre: 467.55 saniye

📁 Dosya başarıyla güncellendi!


In [1]:
import pandas as pd

df = pd.read_csv("test_output_first_1000.csv")

# Status kodlarının frekansı
status_counts = df["status"].value_counts(dropna=False)

print("📊 Status Kodları Dağılımı:")
print(status_counts)

# Ayrıca kaç tane domain hiç çalışmamış (main_url None)
no_url = df[df["main_url"].isna()].shape[0]
print(f"\n❌ Hiç URL bulunamayan domain sayısı: {no_url}")


📊 Status Kodları Dağılımı:
status
200.0    652
NaN      344
202.0      3
204.0      1
Name: count, dtype: int64

❌ Hiç URL bulunamayan domain sayısı: 344


In [2]:
import pandas as pd
import requests
import socket
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# --------------------------------------
# AYARLAR
# --------------------------------------
NUM_THREADS = 40      # biraz düşürdük, sistem daha rahat eder
TIMEOUT = 4
requests.packages.urllib3.disable_warnings()

# --------------------------------------
# DOSYA OKUMA
# --------------------------------------
df = pd.read_csv("test_output_first_1000.csv")

targets = df[df["main_url"].isna() | df["status"].isna()]
domains_to_fix = targets["domain"].tolist()

print(f"🎯 2. tur (hafif) taramaya girecek domain sayısı: {len(domains_to_fix)}\n")


# --------------------------------------
# DNS KONTROLÜ
# --------------------------------------
def dns_exists(domain):
    try:
        socket.gethostbyname(domain)
        return True
    except:
        return False


# --------------------------------------
# ORTA SEVİYE URL VARYASYONLARI
# --------------------------------------
def build_variants(domain):
    # Sıralamayı biraz değiştiriyoruz, belki sadece http çalışanlar vardır
    return [
        f"http://{domain}",
        f"http://{domain}/",
        f"http://www.{domain}",
        f"http://www.{domain}/",
        f"https://{domain}",
        f"https://{domain}/",
        f"https://www.{domain}",
        f"https://www.{domain}/",
        f"http://{domain}/index.html",
        f"https://{domain}/index.html",
    ]


# --------------------------------------
# USER-AGENT
# --------------------------------------
headers = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}


# --------------------------------------
# TARAMA FONKSİYONU
# --------------------------------------
def lite_scan(domain):

    if not dns_exists(domain):
        return domain, None, None

    variants = build_variants(domain)

    for url in variants:
        try:
            # HEAD dene
            r = requests.head(url, timeout=TIMEOUT, verify=False,
                              allow_redirects=True, headers=headers)
            if r.status_code < 400:
                # GET ile doğrula
                try:
                    g = requests.get(url, timeout=TIMEOUT, verify=False,
                                     allow_redirects=True, headers=headers)
                    if g.status_code < 400:
                        return domain, url, g.status_code
                except:
                    pass
        except:
            pass

        # HEAD başarısızsa direkt GET
        try:
            g = requests.get(url, timeout=TIMEOUT, verify=False,
                             allow_redirects=True, headers=headers)
            if g.status_code < 400:
                return domain, url, g.status_code
        except:
            continue

    return domain, None, None


# --------------------------------------
# MULTI-THREAD + PROGRESS LOG
# --------------------------------------
results = {}
start_time = time.time()
done = 0
total = len(domains_to_fix)

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {executor.submit(lite_scan, d): d for d in domains_to_fix}

    for future in as_completed(futures):
        domain, url, status = future.result()
        results[domain] = (url, status)

        done += 1
        if done % 20 == 0:
            print(f"📌 İlerleme: {done}/{total} tamamlandı")

elapsed = time.time() - start_time
print(f"\n⏳ 2. tur (hafif) tarama bitti! Süre: {elapsed:.2f} saniye\n")


# --------------------------------------
# CSV'Yİ YERİNDE GÜNCELLE
# --------------------------------------
for domain, (url, status) in results.items():
    df.loc[df["domain"] == domain, "main_url"] = url
    df.loc[df["domain"] == domain, "status"] = status

df.to_csv("test_output_first_1000.csv", index=False)

print("📁 Dosya 2. tur için başarıyla güncellendi!")


🎯 2. tur (hafif) taramaya girecek domain sayısı: 344

📌 İlerleme: 20/344 tamamlandı
📌 İlerleme: 40/344 tamamlandı
📌 İlerleme: 60/344 tamamlandı
📌 İlerleme: 80/344 tamamlandı
📌 İlerleme: 100/344 tamamlandı
📌 İlerleme: 120/344 tamamlandı
📌 İlerleme: 140/344 tamamlandı
📌 İlerleme: 160/344 tamamlandı
📌 İlerleme: 180/344 tamamlandı
📌 İlerleme: 200/344 tamamlandı
📌 İlerleme: 220/344 tamamlandı
📌 İlerleme: 240/344 tamamlandı
📌 İlerleme: 260/344 tamamlandı
📌 İlerleme: 280/344 tamamlandı
📌 İlerleme: 300/344 tamamlandı
📌 İlerleme: 320/344 tamamlandı
📌 İlerleme: 340/344 tamamlandı

⏳ 2. tur (hafif) tarama bitti! Süre: 477.29 saniye

📁 Dosya 2. tur için başarıyla güncellendi!


In [3]:
import pandas as pd

df = pd.read_csv("test_output_first_1000.csv")
print(df["status"].value_counts(dropna=False))
print("\nNaN sayısı:", df[df["main_url"].isna()].shape[0])


status
200.0    653
NaN      343
202.0      3
204.0      1
Name: count, dtype: int64

NaN sayısı: 343


In [4]:
import pandas as pd
import requests
import socket
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# -----------------------------
# AYARLAR
# -----------------------------
NUM_THREADS = 60
TIMEOUT = 4  # başarı / hız dengesi
CHECKPOINT_SAVE_EVERY = 5000   # kaç domain'de bir CSV kaydı
PROGRESS_PRINT_EVERY = 2000    # kaç domain'de bir durum yazılsın

requests.packages.urllib3.disable_warnings()

# -----------------------------
# DOMAIN LİSTESİNİ YÜKLE
# -----------------------------
df = pd.read_csv("merged_domain_list.csv", header=None, names=["domain"])
df["domain"] = df["domain"].astype(str).str.strip()

# İlk satır header ise at
df = df[df["domain"].str.lower() != "domain"].reset_index(drop=True)

# Eğer daha önce sonuç kolonu yoksa ekle
if "main_url" not in df.columns:
    df["main_url"] = pd.NA
if "status" not in df.columns:
    df["status"] = pd.NA

# Zaten main_url'i bulunanları atla (resume edilebilir)
mask_need_scan = df["main_url"].isna()
domains_to_scan = df.loc[mask_need_scan, "domain"].tolist()

total_to_scan = len(domains_to_scan)
print(f"🚀 Toplam taranacak domain sayısı: {total_to_scan}\n")

if total_to_scan == 0:
    print("Taranacak domain kalmamış görünüyor. Çıkıyorum.")
else:

    # -----------------------------
    # DNS KONTROLÜ (A + AAAA, CNAME transparan)
    # -----------------------------
    def dns_exists(hostname: str) -> bool:
        try:
            # getaddrinfo hem IPv4 hem IPv6'yı dener
            socket.getaddrinfo(hostname, 80)
            return True
        except:
            return False

    # -----------------------------
    # SUBDOMAIN DISCOVERY
    # -----------------------------
    SUB_CANDIDATES = [
        "www", "www2", "portal", "app", "secure",
        "blog", "web", "panel", "login", "store",
        "cpanel", "mail"
    ]

    def normalize_root(domain: str) -> str:
        d = domain.strip().lower()
        if d.startswith("www."):
            d = d[4:]
        return d

    def build_host_candidates(domain: str):
        """
        Önce root ve www/root, başarısız olursa subdomain'leri dener.
        """
        root = normalize_root(domain)
        hosts = []

        # ana adaylar
        hosts.append(root)
        hosts.append(f"www.{root}")

        # subdomain adayları (daha sonra fallback olarak denenir)
        sub_hosts = [f"{sub}.{root}" for sub in SUB_CANDIDATES]

        return hosts, sub_hosts

    # -----------------------------
    # URL VARYASYONLARI
    # -----------------------------
    def build_urls_for_host(host: str):
        urls = []
        schemes = ["https", "http"]
        paths = ["", "/", "/index.html"]
        for sch in schemes:
            for p in paths:
                urls.append(f"{sch}://{host}{p}")
        return urls

    # -----------------------------
    # USER-AGENT
    # -----------------------------
    HEADERS = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    }

    # -----------------------------
    # TEK DOMAIN TARAMA
    # -----------------------------
    def scan_single_domain(domain: str):
        """
        1) root & www için dene
        2) olmazsa subdomain adaylarını dene
        3) DNS yoksa direkt None dön
        """

        primary_hosts, sub_hosts = build_host_candidates(domain)

        # 1) Öncelik root + www
        for host in primary_hosts:
            if not dns_exists(host):
                continue
            for url in build_urls_for_host(host):
                # HEAD
                try:
                    r = requests.head(
                        url,
                        timeout=TIMEOUT,
                        verify=False,
                        allow_redirects=True,
                        headers=HEADERS
                    )
                    if r.status_code < 400:
                        # GET ile doğrula
                        try:
                            g = requests.get(
                                url,
                                timeout=TIMEOUT,
                                verify=False,
                                allow_redirects=True,
                                headers=HEADERS
                            )
                            if g.status_code < 400:
                                return domain, g.url, g.status_code
                        except:
                            pass
                except:
                    pass

                # HEAD başarısızsa direkt GET fallback
                try:
                    g = requests.get(
                        url,
                        timeout=TIMEOUT,
                        verify=False,
                        allow_redirects=True,
                        headers=HEADERS
                    )
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
                except:
                    continue

        # 2) root çalışmadıysa subdomain adaylarını dene
        for host in sub_hosts:
            if not dns_exists(host):
                continue
            for url in build_urls_for_host(host):
                try:
                    r = requests.head(
                        url,
                        timeout=TIMEOUT,
                        verify=False,
                        allow_redirects=True,
                        headers=HEADERS
                    )
                    if r.status_code < 400:
                        try:
                            g = requests.get(
                                url,
                                timeout=TIMEOUT,
                                verify=False,
                                allow_redirects=True,
                                headers=HEADERS
                            )
                            if g.status_code < 400:
                                return domain, g.url, g.status_code
                        except:
                            pass
                except:
                    pass

                try:
                    g = requests.get(
                        url,
                        timeout=TIMEOUT,
                        verify=False,
                        allow_redirects=True,
                        headers=HEADERS
                    )
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
                except:
                    continue

        # Hiçbiri çalışmadıysa
        return domain, None, None

    # -----------------------------
    # MULTI-THREAD TARAMA
    # -----------------------------
    start_time = time.time()
    processed = 0
    total = total_to_scan

    print("🔎 Tarama başlıyor...\n")

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        future_to_domain = {
            executor.submit(scan_single_domain, d): d
            for d in domains_to_scan
        }

        for future in as_completed(future_to_domain):
            domain, url, status = future.result()

            # DataFrame'i yerinde güncelle
            df.loc[df["domain"] == domain, "main_url"] = url
            df.loc[df["domain"] == domain, "status"] = status

            processed += 1

            # PROGRESS PRINT (her 2000 domain'de bir)
            if processed % PROGRESS_PRINT_EVERY == 0 or processed == total:
                found_count = df["main_url"].notna().sum()
                nan_count = df["main_url"].isna().sum()
                elapsed = time.time() - start_time
                print(
                    f"📌 Ara Durum: {processed}/{total} domain işlendi | "
                    f"Bulunan URL: {found_count} | NaN: {nan_count} | "
                    f"Geçen süre: {elapsed:.1f} sn"
                )

            # CHECKPOINT KAYIT (her 5000 domain'de bir)
            if processed % CHECKPOINT_SAVE_EVERY == 0 or processed == total:
                df.to_csv("domain_scan_results.csv", index=False)
                print("💾 Ara kayıt alındı: domain_scan_results.csv\n")

    total_elapsed = time.time() - start_time
    print(f"✅ Tarama bitti! Toplam geçen süre: {total_elapsed:.1f} sn")
    print("📁 Son sonuçlar: domain_scan_results.csv dosyasına kaydedildi.")


🚀 Toplam taranacak domain sayısı: 32705

🔎 Tarama başlıyor...

📌 Ara Durum: 2000/32705 domain işlendi | Bulunan URL: 1502 | NaN: 31203 | Geçen süre: 1902.3 sn
📌 Ara Durum: 4000/32705 domain işlendi | Bulunan URL: 2931 | NaN: 29774 | Geçen süre: 3784.8 sn
💾 Ara kayıt alındı: domain_scan_results.csv

📌 Ara Durum: 6000/32705 domain işlendi | Bulunan URL: 4314 | NaN: 28391 | Geçen süre: 5868.2 sn
📌 Ara Durum: 8000/32705 domain işlendi | Bulunan URL: 5779 | NaN: 26926 | Geçen süre: 7875.2 sn
📌 Ara Durum: 10000/32705 domain işlendi | Bulunan URL: 7365 | NaN: 25340 | Geçen süre: 9747.5 sn
💾 Ara kayıt alındı: domain_scan_results.csv

📌 Ara Durum: 12000/32705 domain işlendi | Bulunan URL: 8868 | NaN: 23837 | Geçen süre: 11524.7 sn
📌 Ara Durum: 14000/32705 domain işlendi | Bulunan URL: 10268 | NaN: 22437 | Geçen süre: 13594.6 sn
💾 Ara kayıt alındı: domain_scan_results.csv

📌 Ara Durum: 16000/32705 domain işlendi | Bulunan URL: 11652 | NaN: 21053 | Geçen süre: 15983.8 sn
📌 Ara Durum: 18000/32705 

In [1]:
import pandas as pd

df = pd.read_csv("domain_scan_results.csv")
print(df["status"].value_counts(dropna=False))
print("\nNaN sayısı:", df[df["main_url"].isna()].shape[0])


status
200.0    25427
NaN       7169
202.0       52
204.0       33
307.0       14
301.0        3
103.0        2
247.0        2
302.0        2
206.0        1
Name: count, dtype: int64

NaN sayısı: 7169


In [2]:
import pandas as pd
import math

# Ayar — bir parçada kaç domain olsun?
CHUNK_SIZE = 50   # istersen 20, 100, 200 yapabilirsin

# Dosyayı yükle
df = pd.read_csv("domain_scan_results.csv")

# NaN olan domainleri seç
nan_domains = df[df["main_url"].isna()]["domain"].tolist()

print(f"Toplam NaN domain: {len(nan_domains)}\n")

# Parça sayısını hesapla
num_chunks = math.ceil(len(nan_domains) / CHUNK_SIZE)

print(f"Toplam parça sayısı: {num_chunks}\n")

# Parçalara ayır
for i in range(num_chunks):
    chunk = nan_domains[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE]

    # Dosyaya kaydet
    filename = f"nan_domains_part_{i+1}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        for d in chunk:
            f.write(d + "\n")

    print(f"📁 {filename} oluşturuldu — {len(chunk)} domain içeriyor")

print("\n✔ İşlem tamamlandı! Artık ChatGPT’ye her parçayı gönderebilirsin.")


Toplam NaN domain: 7169

Toplam parça sayısı: 144

📁 nan_domains_part_1.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_2.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_3.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_4.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_5.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_6.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_7.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_8.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_9.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_10.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_11.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_12.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_13.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_14.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_15.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_part_16.txt oluşturuldu — 50 domain içeriyor
📁 nan_domains_

In [5]:
import pandas as pd

df = pd.read_csv("domain_scan_results.csv")
print(df["status"].value_counts(dropna=False))
print("\nNaN sayısı:", df[df["main_url"].isna()].shape[0])


status
200.0    25571
NaN       7017
202.0       52
204.0       33
307.0       14
301.0       11
103.0        2
247.0        2
302.0        2
206.0        1
Name: count, dtype: int64

NaN sayısı: 7017


In [6]:
import pandas as pd

# Ana dosyayı yükle
df = pd.read_csv("domain_scan_results.csv")

# Status kodunu numerik olarak al (NaN vs. 200 sorun çıkmasın)
df["status"] = pd.to_numeric(df["status"], errors="coerce")

# 200 olanlar
df_success = df[df["status"] == 200]

# 200 olmayanlar veya boş olanlar
df_other = df[df["status"] != 200]

# CSV'lere kaydet
df_success.to_csv("success_200.csv", index=False)
df_other.to_csv("others.csv", index=False)

print("✔ success_200.csv oluşturuldu")
print("✔ others.csv oluşturuldu")
print(f"200 OK olan domain sayısı: {df_success.shape[0]}")
print(f"Kalan domain sayısı: {df_other.shape[0]}")


✔ success_200.csv oluşturuldu
✔ others.csv oluşturuldu
200 OK olan domain sayısı: 25601
Kalan domain sayısı: 7104


In [7]:
import pandas as pd

df = pd.read_csv("success_200.csv")
print(df["status"].value_counts(dropna=False))
print("\nNaN sayısı:", df[df["main_url"].isna()].shape[0])

status
200.0    25601
Name: count, dtype: int64

NaN sayısı: 0


In [9]:
import os

print(os.getcwd())  # Şu anki çalışma dizini
print(os.listdir())  # Bu dizindeki klasörler



C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma
['.ipynb_checkpoints', 'Beningdata', 'bp_cleaned.csv', 'chromedriver-win64', 'CreateBegingData.ipynb', 'CreatePhishData.ipynb', 'DATAPRE.ipynb', 'domain_scan_results.csv', 'duplicate_counts.csv', 'duplicate_main_urls.csv', 'merged_domain_list.csv', 'others.csv', 'phishdata', 'report_single_url_domains.csv', 'success_200.csv', 'success_200_unique.csv', 'test_output_first_1000.csv', 'Untitled.ipynb', 'Untitled1.ipynb', 'Url_version1']


In [10]:
import os

target = "top-1m.csv"

for root, dirs, files in os.walk(".", topdown=True):
    if target in files:
        print("Bulundu:", os.path.join(root, target))
        break


Bulundu: .\Beningdata\top-1m.csv


In [12]:
import pandas as pd

# Bulduğumuz gerçek dosya yolu
input_path = "./Beningdata/top-1m.csv"
output_path = "./Beningdata/top-50k.csv"

# CSV'yi oku
df = pd.read_csv(input_path, header=None, names=["rank", "domain"])

# İlk 50.000 domaini al
df_50k = df.head(50000)

# Yeni CSV dosyasına kaydet
df_50k.to_csv(output_path, index=False)

print("📁 İşlem tamamlandı!")
print("Toplam alınan domain:", len(df_50k))
print("Kayıt edilen dosya:", output_path)


📁 İşlem tamamlandı!
Toplam alınan domain: 50000
Kayıt edilen dosya: ./Beningdata/top-50k.csv


In [13]:
import pandas as pd

# Dosya yolları
top50_path = "./Beningdata/top-50k.csv"
merged_path = "merged_domain_list.csv"
output_path = "./Beningdata/top-50k_cleaned.csv"

# Dosyaları oku
df_top50 = pd.read_csv(top50_path)
df_merged = pd.read_csv(merged_path)

# Domain kolonlarını normalize et (küçük harf, boşluk silme)
df_top50["domain"] = df_top50["domain"].astype(str).str.strip().str.lower()
df_merged["domain"] = df_merged["domain"].astype(str).str.strip().str.lower()

# Duplicate olan domainleri çıkar
df_cleaned = df_top50[~df_top50["domain"].isin(df_merged["domain"])]

# Temiz listeyi kaydet
df_cleaned.to_csv(output_path, index=False)

print("📁 İşlem tamamlandı!")
print("Orijinal top-50k:", len(df_top50))
print("Merged liste:", len(df_merged))
print("Kalan domain:", len(df_cleaned))
print("Kayıt edilen dosya:", output_path)


📁 İşlem tamamlandı!
Orijinal top-50k: 50000
Merged liste: 32705
Kalan domain: 24445
Kayıt edilen dosya: ./Beningdata/top-50k_cleaned.csv


In [14]:
import pandas as pd
import requests
import socket
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# -----------------------------
# AYARLAR
# -----------------------------
NUM_THREADS = 60
TIMEOUT = 4
CHECKPOINT_SAVE_EVERY = 5000
PROGRESS_PRINT_EVERY = 2000

requests.packages.urllib3.disable_warnings()

# -----------------------------
# DOMAIN LİSTESİNİ YÜKLE
# -----------------------------
input_path = "./Beningdata/top-50k_cleaned.csv"
output_path = "./Beningdata/new_url.csv"

df = pd.read_csv(input_path)
df["domain"] = df["domain"].astype(str).str.strip()

# Eğer main_url / status kolonları yoksa oluştur
if "main_url" not in df.columns:
    df["main_url"] = pd.NA
if "status" not in df.columns:
    df["status"] = pd.NA

# Zaten işlenmişleri atla (resume için)
mask_need_scan = df["main_url"].isna()
domains_to_scan = df.loc[mask_need_scan, "domain"].tolist()

total_to_scan = len(domains_to_scan)
print(f"🚀 Toplam taranacak domain sayısı: {total_to_scan}\n")

if total_to_scan == 0:
    print("Taranacak domain yok. Çıkılıyor.")
    df.to_csv(output_path, index=False)
else:

    # -----------------------------
    # DNS KONTROLÜ
    # -----------------------------
    def dns_exists(hostname: str) -> bool:
        try:
            socket.getaddrinfo(hostname, 80)
            return True
        except:
            return False

    # -----------------------------
    # SUBDOMAIN CANDIDATES
    # -----------------------------
    SUB_CANDIDATES = [
        "www", "www2", "portal", "app", "secure",
        "blog", "web", "panel", "login", "store",
        "cpanel", "mail"
    ]

    def normalize_root(domain: str) -> str:
        d = domain.strip().lower()
        if d.startswith("www."):
            d = d[4:]
        return d

    def build_host_candidates(domain: str):
        root = normalize_root(domain)
        hosts = [root, f"www.{root}"]
        sub_hosts = [f"{sub}.{root}" for sub in SUB_CANDIDATES]
        return hosts, sub_hosts

    # -----------------------------
    # URL oluşturma
    # -----------------------------
    def build_urls_for_host(host: str):
        urls = []
        schemes = ["https", "http"]
        paths = ["", "/", "/index.html"]
        for sch in schemes:
            for p in paths:
                urls.append(f"{sch}://{host}{p}")
        return urls

    HEADERS = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    }

    # -----------------------------
    # TEK DOMAIN TARAMA
    # -----------------------------
    def scan_single_domain(domain: str):
        primary_hosts, sub_hosts = build_host_candidates(domain)

        # 1) Ana hostlar
        for host in primary_hosts:
            if not dns_exists(host):
                continue
            for url in build_urls_for_host(host):
                # HEAD dene
                try:
                    r = requests.head(url, timeout=TIMEOUT, verify=False,
                                      allow_redirects=True, headers=HEADERS)
                    if r.status_code < 400:
                        try:
                            g = requests.get(url, timeout=TIMEOUT, verify=False,
                                             allow_redirects=True, headers=HEADERS)
                            if g.status_code < 400:
                                return domain, g.url, g.status_code
                        except:
                            pass
                except:
                    pass

                # Direkt GET fallback
                try:
                    g = requests.get(url, timeout=TIMEOUT, verify=False,
                                     allow_redirects=True, headers=HEADERS)
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
                except:
                    continue

        # 2) Subdomain fallback
        for host in sub_hosts:
            if not dns_exists(host):
                continue
            for url in build_urls_for_host(host):
                try:
                    r = requests.head(url, timeout=TIMEOUT, verify=False,
                                      allow_redirects=True, headers=HEADERS)
                    if r.status_code < 400:
                        try:
                            g = requests.get(url, timeout=TIMEOUT, verify=False,
                                             allow_redirects=True, headers=HEADERS)
                            if g.status_code < 400:
                                return domain, g.url, g.status_code
                        except:
                            pass
                except:
                    pass

                try:
                    g = requests.get(url, timeout=TIMEOUT, verify=False,
                                     allow_redirects=True, headers=HEADERS)
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
                except:
                    continue

        return domain, None, None

    # -----------------------------
    # MULTI-THREAD TARAMA
    # -----------------------------
    start_time = time.time()
    processed = 0
    total = total_to_scan

    print("🔎 Tarama başlıyor...\n")

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        future_to_domain = {
            executor.submit(scan_single_domain, d): d
            for d in domains_to_scan
        }

        for future in as_completed(future_to_domain):
            domain, url, status = future.result()

            df.loc[df["domain"] == domain, "main_url"] = url
            df.loc[df["domain"] == domain, "status"] = status

            processed += 1

            # Durum yazdır
            if processed % PROGRESS_PRINT_EVERY == 0:
                print(f"📌 {processed}/{total} tamamlandı")

            # Ara kayıt
            if processed % CHECKPOINT_SAVE_EVERY == 0:
                df.to_csv(output_path, index=False)
                print(f"💾 Ara kayıt alındı: {output_path}")

    # Final kayıt
    df.to_csv(output_path, index=False)
    total_elapsed = time.time() - start_time

    print(f"\n✅ Tarama tamamlandı! Süre: {total_elapsed:.1f} sn")
    print(f"📁 Son çıktı: {output_path}")


🚀 Toplam taranacak domain sayısı: 24445

🔎 Tarama başlıyor...

📌 2000/24445 tamamlandı
📌 4000/24445 tamamlandı
💾 Ara kayıt alındı: ./Beningdata/new_url.csv
📌 6000/24445 tamamlandı
📌 8000/24445 tamamlandı
📌 10000/24445 tamamlandı
💾 Ara kayıt alındı: ./Beningdata/new_url.csv
📌 12000/24445 tamamlandı
📌 14000/24445 tamamlandı
💾 Ara kayıt alındı: ./Beningdata/new_url.csv
📌 16000/24445 tamamlandı
📌 18000/24445 tamamlandı
📌 20000/24445 tamamlandı
💾 Ara kayıt alındı: ./Beningdata/new_url.csv
📌 22000/24445 tamamlandı
📌 24000/24445 tamamlandı

✅ Tarama tamamlandı! Süre: 29161.0 sn
📁 Son çıktı: ./Beningdata/new_url.csv


In [4]:
import os
print(os.getcwd())


C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma


In [5]:
import pandas as pd

input_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top-1m.csv"
output_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv"

df = pd.read_csv(input_path, header=None, names=["rank", "domain"], nrows=85000)

df = df[["domain"]]

df.to_csv(output_path, index=False)

print("İlk 85.000 domain başarıyla kaydedildi!")


İlk 85.000 domain başarıyla kaydedildi!


In [9]:
import pandas as pd
from urllib.parse import urlparse

# Dosya yolları
main_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv"
remove_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\new_url.csv"

# Ana dosya
df_main = pd.read_csv(main_path)

# Çıkarılacak domainler
df_remove = pd.read_csv(remove_path)

# --- Normalize domain fonksiyonu ---
def normalize_domain(x):
    x = str(x).strip().lower()

    # Eğer URL ise
    if "://" in x:
        x = urlparse(x).netloc

    # www. temizle
    if x.startswith("www."):
        x = x[4:]

    return x

# Ana dosya normalize
df_main["domain_norm"] = df_main["domain"].apply(normalize_domain)

# Çıkarılacak domainleri normalize
df_remove["domain_norm"] = df_remove["domain"].apply(normalize_domain)

# Filtreleme
df_filtered = df_main[~df_main["domain_norm"].isin(df_remove["domain_norm"])]

# Ana dosya üzerine yaz (overwrite)
df_filtered[["domain"]].to_csv(main_path, index=False)

print("Gerçek filtreleme tamamlandı!")
print("Silinen domain sayısı:", len(df_main) - len(df_filtered))
print("Kalan domain:", len(df_filtered))


Gerçek filtreleme tamamlandı!
Silinen domain sayısı: 24445
Kalan domain: 60555


In [ ]:
Bitirme_Projesi_Dataset_Olusturma/bp_cleaned.csv

In [15]:
import pandas as pd
from urllib.parse import urlparse

# === Dosya yolları ===
main_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv"
scan_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\domain_scan_results.csv"

# Ana domain listesi
df_main = pd.read_csv(main_path)

# Çıkarılacak domainlerin olduğu dosya
df_scan = pd.read_csv(scan_path)

# --- Normalize domain fonksiyonu ---
def normalize_domain(x):
    x = str(x).strip().lower()

    # Eğer URL ise domaini çıkar
    if "://" in x:
        x = urlparse(x).netloc

    # www. kaldır
    if x.startswith("www."):
        x = x[4:]

    return x


# === domain_scan_results.csv için DOMAIN SÜTUNUNU OTOMATİK BUL ===
possible_domain_cols = ["domain", "main_url", "url", "website"]

domain_col = None
for col in df_scan.columns:
    if col.lower() in possible_domain_cols:
        domain_col = col
        break

# Eğer domain kolonu yoksa → ilk URL benzeri sütunu seç
if domain_col is None:
    for col in df_scan.columns:
        if df_scan[col].astype(str).str.contains("\.").any():
            domain_col = col
            break

print(f"Kullanılan domain sütunu: {domain_col}")

# === Normalize ederek domaini çıkar ===
df_main["domain_norm"] = df_main["domain"].apply(normalize_domain)
df_scan["domain_norm"] = df_scan[domain_col].apply(normalize_domain)

# === Filtreleme ===
df_filtered = df_main[~df_main["domain_norm"].isin(df_scan["domain_norm"])]

# === Üzerine yaz (overwrite) ===
df_filtered[["domain"]].to_csv(main_path, index=False)

print("domain_scan_results.csv içindeki domainler çıkarıldı!")
print("Silinen domain:", len(df_main) - len(df_filtered))
print("Kalan domain:", len(df_filtered))



Kullanılan domain sütunu: domain
domain_scan_results.csv içindeki domainler çıkarıldı!
Silinen domain: 22737
Kalan domain: 34342


In [20]:
import pandas as pd

# Dosya yolları
main_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv"
tranco_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata\top-1m.csv"

# Mevcut domain listesi
df_main = pd.read_csv(main_path)

# Tranco dosyası
df_tranco = pd.read_csv(
    tranco_path,
    header=None,
    names=["rank", "domain"]
)

# 85001–95000 arası slice
df_slice = df_tranco[(df_tranco["rank"] >= 95001) & (df_tranco["rank"] <= 110000)]

# Sadece domain kolonunu al
df_slice = df_slice[["domain"]]

# Mevcut liste ile birleştir (append)
df_final = pd.concat([df_main, df_slice], ignore_index=True)

# Duplicate domain'leri temizle
df_final = df_final.drop_duplicates(subset=["domain"])

# Ana dosya üzerine yaz (overwrite)
df_final.to_csv(main_path, index=False)

print("Yeni Tranco domainleri eklendi!")
print("Eklenen domain sayısı:", len(df_slice))
print("Toplam yeni domain sayısı:", len(df_final))
print(f"Güncellenen dosya: {main_path}")


Yeni Tranco domainleri eklendi!
Eklenen domain sayısı: 15000
Toplam yeni domain sayısı: 59251
Güncellenen dosya: C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv


In [21]:
import os

folder = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Beningdata"
print(os.listdir(folder))


['.ipynb_checkpoints', 'bp.txt', 'bp_cleaned-Copy1.csv', 'failed_urls.csv', 'first_url.csv', 'new_url.csv', 'real_paths_master.csv', 'success_200_unique.csv', 'top-1m.csv', 'top-50k.csv', 'top-50k_cleaned.csv', 'top_25000.csv']


In [22]:
import pandas as pd
from urllib.parse import urlparse

# Dosya yolları
main_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\Lutfen_son_domain_olsun_1.csv"
remove_path = r"C:\Users\elifo\Bitirme_Projesi_Dataset_Olusturma\TurkVerisiTamDomain.csv"

# Ana dosya
df_main = pd.read_csv(main_path)

# Çıkarılacak domainler (bp.csv)
df_remove = pd.read_csv(remove_path)

# --- Normalize domain fonksiyonu ---
def normalize_domain(x):
    x = str(x).strip().lower()

    # Eğer URL ise
    if "://" in x:
        x = urlparse(x).netloc

    # www. kaldır
    if x.startswith("www."):
        x = x[4:]

    return x

# Ana dosya normalize (domain kolonu)
df_main["domain_norm"] = df_main["domain"].apply(normalize_domain)

# Remove dosyası normalize (deger kolonu)
df_remove["domain_norm"] = df_remove["deger"].apply(normalize_domain)

# Filtreleme
df_filtered = df_main[~df_main["domain_norm"].isin(df_remove["domain_norm"])]

# Ana dosyanın üzerine yaz (overwrite)
df_filtered[["domain"]].to_csv(main_path, index=False)

print("Gerçek filtreleme tamamlandı!")
print("Silinen domain:", len(df_main) - len(df_filtered))
print("Kalan domain:", len(df_filtered))


Gerçek filtreleme tamamlandı!
Silinen domain: 127
Kalan domain: 59124


In [1]:
import pandas as pd
import requests
import socket
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import sys
import os

# -----------------------------
# AYARLAR
# -----------------------------
NUM_THREADS = 60
TIMEOUT = 4
CHECKPOINT_SAVE_EVERY = 500
PROGRESS_PRINT_EVERY = 200

requests.packages.urllib3.disable_warnings()

# -----------------------------
# DOSYA YOLLARI (part1)
# -----------------------------
MAIN_FILE = "Lutfen_son_domain_olsun_1_part1.csv"
OUTPUT_FILE = "domain_scan_results_part1.csv"

# -----------------------------
# DOMAIN LİSTESİNİ YÜKLE
# -----------------------------
if not os.path.exists(MAIN_FILE):
    print(f"HATA: {MAIN_FILE} bulunamadı!")
    sys.exit(1)

df = pd.read_csv(MAIN_FILE, header=None, names=["domain"])
df["domain"] = df["domain"].astype(str).str.strip()


df = df[df["domain"].str.lower() != "domain"].reset_index(drop=True)

if "main_url" not in df.columns:
    df["main_url"] = pd.NA
if "status" not in df.columns:
    df["status"] = pd.NA

mask = df["main_url"].isna()
domains_to_scan = df.loc[mask, "domain"].tolist()
total = len(domains_to_scan)

print(f"Taranacak domain sayısı (part1): {total}")

if total == 0:
    sys.exit(0)

# DNS test
def dns_exists(host):
    try:
        socket.getaddrinfo(host, 80)
        return True
    except:
        return False

SUBS = ["www", "www2", "portal", "app", "secure", "blog", "web", "panel", "login", "store", "cpanel", "mail"]

def normalize(root):
    r = root.lower().strip()
    if r.startswith("www."):
        r = r[4:]
    return r

def host_candidates(domain):
    root = normalize(domain)
    base = [root, f"www.{root}"]
    subs = [f"{s}.{root}" for s in SUBS]
    return base, subs

def build_urls(host):
    urls = []
    for sch in ["https", "http"]:
        for p in ["", "/", "/index.html"]:
            urls.append(f"{sch}://{host}{p}")
    return urls

HEAD = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def scan_single(domain):
    base, subs = host_candidates(domain)

    for host in base:
        if not dns_exists(host):
            continue
        for url in build_urls(host):
            try:
                r = requests.head(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                if r.status_code < 400:
                    g = requests.get(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
            except:
                pass
            try:
                g = requests.get(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                if g.status_code < 400:
                    return domain, g.url, g.status_code
            except:
                pass

    for host in subs:
        if not dns_exists(host):
            continue
        for url in build_urls(host):
            try:
                r = requests.head(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                if r.status_code < 400:
                    g = requests.get(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                    if g.status_code < 400:
                        return domain, g.url, g.status_code
            except:
                pass
            try:
                g = requests.get(url, timeout=TIMEOUT, verify=False, allow_redirects=True, headers=HEAD)
                if g.status_code < 400:
                    return domain, g.url, g.status_code
            except:
                pass

    return domain, None, None

processed = 0
start = time.time()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as ex:
    futures = {ex.submit(scan_single, d): d for d in domains_to_scan}

    for f in as_completed(futures):
        domain, url, status = f.result()
        df.loc[df["domain"] == domain, ["main_url", "status"]] = [url, status]

        processed += 1

        if processed % PROGRESS_PRINT_EVERY == 0:
            print(f"Part1 -> {processed}/{total}")

        if processed % CHECKPOINT_SAVE_EVERY == 0:
            df.to_csv(OUTPUT_FILE, index=False)
            print("Ara kayıt (part1) alındı.")

df.to_csv(OUTPUT_FILE, index=False)
print("Part1 tamamlandı.")


Taranacak domain sayısı (part1): 29561
Part1 -> 200/29561
Part1 -> 400/29561
Ara kayıt (part1) alındı.
Part1 -> 600/29561
Part1 -> 800/29561
Part1 -> 1000/29561
Ara kayıt (part1) alındı.
Part1 -> 1200/29561
Part1 -> 1400/29561
Ara kayıt (part1) alındı.
Part1 -> 1600/29561
Part1 -> 1800/29561
Part1 -> 2000/29561
Ara kayıt (part1) alındı.
Part1 -> 2200/29561
Part1 -> 2400/29561
Ara kayıt (part1) alındı.
Part1 -> 2600/29561
Part1 -> 2800/29561
Part1 -> 3000/29561
Ara kayıt (part1) alındı.
Part1 -> 3200/29561
Part1 -> 3400/29561
Ara kayıt (part1) alındı.
Part1 -> 3600/29561
Part1 -> 3800/29561
Part1 -> 4000/29561
Ara kayıt (part1) alındı.
Part1 -> 4200/29561
Part1 -> 4400/29561
Ara kayıt (part1) alındı.
Part1 -> 4600/29561
Part1 -> 4800/29561
Part1 -> 5000/29561
Ara kayıt (part1) alındı.
Part1 -> 5200/29561
Part1 -> 5400/29561
Ara kayıt (part1) alındı.
Part1 -> 5600/29561
Part1 -> 5800/29561
Part1 -> 6000/29561
Ara kayıt (part1) alındı.
Part1 -> 6200/29561
Part1 -> 6400/29561
Ara kayıt (pa

In [3]:
import pandas as pd

# CSV dosyasını yükle
df = pd.read_csv("domain_scan_results_part1.csv")

# 1) main_url sütununda kaç adet NaN var?
nan_main_url = df["main_url"].isna().sum()

# 2) status_code sütununda kaç adet NaN var?
nan_status_code = df["status"].isna().sum()

# 3) status_code’un aldığı tüm benzersiz değerler ve frekansları
status_code_values = df["status"].value_counts(dropna=False)

print("main_url NaN sayısı:", nan_main_url)
print("status_code NaN sayısı:", nan_status_code)
print("\nstatus_code değerleri ve frekansları:\n", status_code_values)


main_url NaN sayısı: 7084
status_code NaN sayısı: 7084

status_code değerleri ve frekansları:
 status
200.0    22352
NaN       7084
202.0       89
204.0       14
307.0       11
301.0        3
201.0        2
101.0        1
247.0        1
399.0        1
214.0        1
302.0        1
304.0        1
Name: count, dtype: int64


In [5]:
import pandas as pd

# Veriyi yükle
df = pd.read_csv("domain_scan_results_part1.csv")

# 1) main_url veya status_code NaN olan satırları sil
df = df.dropna(subset=["main_url", "status"])

# status_code'u float'tan int'e dönüştür (kolay filtre için)
df["status"] = df["status"].astype(float).astype(int)

# 2) Sadece 2xx (200–299) arası status_code değerlerini koru
df_clean = df[(df["status"] >= 200) & (df["status"] < 300)]

# Kayıt et
df_clean.to_csv("domain_scan_results_part1_cleaned.csv", index=False)

print("Temizleme tamamlandı!")
print("Kalan satır sayısı:", len(df_clean))
print("Kayıt edilen dosya: domain_scan_results_part1_cleaned.csv")


Temizleme tamamlandı!
Kalan satır sayısı: 22459
Kayıt edilen dosya: domain_scan_results_part1_cleaned.csv


In [2]:
import pandas as pd

# Dosya yolları
file1 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths.csv"
file2 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths_2.csv"
file3 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths_3.csv"

# CSV'leri yükle
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)

# Sadece current_url kolonunu çek
urls1 = df1["current_url"]
urls2 = df2["current_url"]
urls3 = df3["current_url"]

# Hepsini birleştir
final_df = pd.concat([urls1, urls2, urls3], ignore_index=True)

# Duplicate temizle
final_df = final_df.drop_duplicates().reset_index(drop=True)

# Kolon adını düzgün yap
final_df = final_df.to_frame(name="current_url")

# CSV olarak kaydet
output_path = "final_benign_labeled.csv"
final_df.to_csv(output_path, index=False)

print("İşlem tamamlandı!")
print(f"Toplam benzersiz current_url sayısı: {len(final_df)}")
print(f"Kayıt edilen dosya: {output_path}")


İşlem tamamlandı!
Toplam benzersiz current_url sayısı: 321707
Kayıt edilen dosya: final_benign_labeled.csv


In [4]:
import pandas as pd

# Dosya yolları
file1 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths.csv"
file2 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths_2.csv"
file3 = "~/Bitirme_Projesi_Dataset_Olusturma/Beningdata/smart_selected_paths_3.csv"

# CSV'leri yükle
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)

# Sadece current_url kolonunu çek
urls1 = df1["current_url"]
urls2 = df2["current_url"]
urls3 = df3["current_url"]

# Hepsini birleştir, hangi dosyadan geldiğini takip etmek için
all_urls = pd.concat(
    [
        urls1.to_frame(name="current_url").assign(source="file1"),
        urls2.to_frame(name="current_url").assign(source="file2"),
        urls3.to_frame(name="current_url").assign(source="file3"),
    ],
    ignore_index=True
)

# Duplicate olan URL'leri bul
duplicate_urls = all_urls[all_urls.duplicated(subset=["current_url"], keep=False)]

# URL'leri gruplayarak listeli şekilde göster
duplicate_list = duplicate_urls.sort_values("current_url")

print("Toplam duplicate current_url sayısı:", duplicate_list["current_url"].nunique())
duplicate_list


Toplam duplicate current_url sayısı: 5989


,current_url,source
215181,http://aws.amazon.com/privacy/aws-privacy-prio...,file3
4348,http://aws.amazon.com/privacy/aws-privacy-prio...,file1
4360,http://aws.amazon.com/privacy/aws-privacy-prio...,file1
215215,http://aws.amazon.com/privacy/aws-privacy-prio...,file3
231598,http://icmregistry.biz/accredited-registrars/,file3
...,...,...
102717,https://zomro.com/rus/website-builder,file1
102712,https://zomro.com/ua/vps,file1
209915,https://zomro.com/ua/vps,file2
329085,https://zomro.com/ua/website-builder,file3


In [6]:
import pandas as pd

# Final dosya
final_file = "final_benign_labeled.csv"
df_final = pd.read_csv(final_file)

# no_url dosyaları
no2 = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final_2.csv"
no3 = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final_3.csv"
no_empty = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final.csv"

# Dosyaları yükle
df_no2 = pd.read_csv(no2)
df_no3 = pd.read_csv(no3)
df_no_empty = pd.read_csv(no_empty)

# Tüm main_url kolonlarını topla
main_urls = pd.concat([
    df_no2["main_url"],
    df_no3["main_url"],
    df_no_empty["main_url"]
], ignore_index=True)

# Duplicate temizle
main_urls = main_urls.drop_duplicates().reset_index(drop=True)

# Final CSV'ye ekle
df_final["main_url"] = main_urls

print("Final satır sayısı:", len(df_final))
print("Eklenen benzersiz main_url sayısı:", len(main_urls))

# Duplicate kontrolü (current_url + main_url)
duplicates = df_final[df_final.duplicated(subset=["current_url", "main_url"], keep=False)]
print("Duplicate kayıt sayısı:", len(duplicates))

# Üzerine yaz
df_final.to_csv(final_file, index=False)



C:\Users\elifo\AppData\Local\Temp\ipykernel_2512\3243774765.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv(final_file)


Final satır sayısı: 321707
Eklenen benzersiz main_url sayısı: 17374
Duplicate kayıt sayısı: 0


In [14]:
import pandas as pd

file_path = "~/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv"

# CSV'yi oku
df = pd.read_csv(file_path)

# Eğer main_url kolonu varsa sil
if "main_url" in df.columns:
    df = df.drop(columns=["main_url"])
    print("main_url sütunu silindi.")
else:
    print("main_url sütunu zaten yok.")

# Dosyayı aynı isimle kaydet (overwrite)
df.to_csv(file_path, index=False)

print("Dosya güncellendi:", file_path)


C:\Users\elifo\AppData\Local\Temp\ipykernel_2512\2785072371.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


main_url sütunu silindi.
Dosya güncellendi: ~/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv


In [15]:
import pandas as pd
import os

# Senin verdiğin path'ler birebir:
file2 = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final_2.csv"
file3 = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final_3.csv"
file_empty = "~/Bitirme_Projesi_Dataset_Olusturma/no_urls_normal_playwright_final.csv"
final_file = "~/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv"

# ~ işaretini Windows home dizinine dönüştür
file2 = os.path.expanduser(file2)
file3 = os.path.expanduser(file3)
file_empty = os.path.expanduser(file_empty)
final_file = os.path.expanduser(final_file)

# Final benign dosyasını yükle
df_final = pd.read_csv(final_file)

# Üç dosyayı oku
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df_empty = pd.read_csv(file_empty)

# Üç dosyadaki main_url sütunlarını al
urls2 = df2["main_url"].dropna()
urls3 = df3["main_url"].dropna()
urls_empty = df_empty["main_url"].dropna()

# Final current_url ile birleştir
combined = pd.concat([
    df_final["current_url"],
    urls2,
    urls3,
    urls_empty
], ignore_index=True)

# Duplicate temizle
combined = combined.drop_duplicates().reset_index(drop=True)

# Final dosyayı yeniden oluştur
df_final = pd.DataFrame({"current_url": combined})

# Final dosyanın üzerine yaz
df_final.to_csv(final_file, index=False)

print("İşlem tamamlandı!")
print("Toplam benzersiz URL:", len(df_final))
print("Güncellenen dosya:", final_file)


İşlem tamamlandı!
Toplam benzersiz URL: 339076
Güncellenen dosya: C:\Users\elifo/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv


In [16]:
import pandas as pd
import os

# Final dosya yolu (senin verdiğin)
final_file = "~/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv"

# ~ dönüştürme
final_file = os.path.expanduser(final_file)

# CSV yükle
df = pd.read_csv(final_file)

# Label sütunu ekle (tüm değerler 0)
df["label"] = 0

# Dosyayı yeniden kaydet
df.to_csv(final_file, index=False)

print("label sütunu eklendi ve tüm değerler 0 yapıldı!")
print("Toplam satır:", len(df))
print("Güncellenen dosya:", final_file)


label sütunu eklendi ve tüm değerler 0 yapıldı!
Toplam satır: 339076
Güncellenen dosya: C:\Users\elifo/Bitirme_Projesi_Dataset_Olusturma/final_benign_labeled.csv
